# DTD: ablation text cho query và candidate pool

Notebook được tách từ `mtacs-train-captions-vitl16-dtd.ipynb`. Giữ nguyên dataset, ViT-L/16,
preprocessing, candidate pool 20% train và captioner hiện tại; bổ sung caption của candidate vào
selector/fusion để kiểm tra vai trò của text cho cả query và candidate. Bản nguồn chỉ fuse caption query.
Mọi cấu hình dùng cùng một lớp model, classifier head, khởi tạo, ngân sách train và pool; nhánh text
bị tắt bằng mask. `visual_only` là baseline cùng kiến trúc cho ablation này, không phải tái lập chính xác
mọi lựa chọn của paper TACS.

DTD có 47 lớp, 40 ảnh/lớp cho mỗi train, validation, test trong một official split: 1.880/1.880/1.880.
Candidate pool gồm 376 ảnh chỉ từ train; query nằm trong pool bị loại khỏi chính danh sách candidate của nó.
Nguồn split: https://www.robots.ox.ac.uk/~vgg/data/dtd/. Một lần chạy chỉ là một split; so sánh benchmark
10 splits cần train độc lập split 1..10 và tổng hợp mean/std, không gộp các split vào một tập train.

Không thay prompt/model sinh caption ở thí nghiệm này. Ưu tiên tái sử dụng `captions.csv` DTD cũ;
tự tạo/resume CSV nếu còn thiếu khi `GENERATE_CAPTIONS_IF_MISSING=True` (mặc định). Model sinh caption nhận pixels, không nhận
label, tên file hay đường dẫn ảnh. Các từ chỉ lớp được dùng cho audit/redaction toàn bộ 47 lớp, không dùng
nhãn thật của từng ảnh để xây caption. Không đưa label candidate vào selector hoặc fusion.

## Các thí nghiệm đã định trước

| Cấu hình | Text query vào fusion | Text candidate vào fusion | Text vào selector | Caption |
|---|---|---|---|---|
| `visual_only` | Không | Không | Không | Nhánh text bị mask |
| `query_only` | Có | Không | Không | Screened |
| `candidate_only` | Không | Có | Không | Screened |
| `both_fusion` | Có | Có | Không | Screened |
| `full_both` | Có | Có | Có | Screened |
| `full_shuffle_both` | Có | Có | Có | Đảo caption cả hai vai trò, giữ mask |
| `full_unscreened` | Có | Có | Có | Caption trước lọc, đối chứng shortcut |
| `full_redacted` | Có | Có | Có | Che tên lớp/alias trong cả query và candidate |

`query_only`/`candidate_only`/`both_fusion` tách vai trò text downstream trong cùng họ selector thị giác.
`full_both` so với `both_fusion` kiểm tra đóng góp thêm của semantic selector. Các model train độc lập có thể
học cách chọn ảnh khác nhau; cell intervention giữ nguyên ảnh context trên cùng checkpoint để tách riêng ảnh hưởng fusion.

`screened` dùng nguyên caption đã tạo, mask CSV, lexical check và ngưỡng CLIP cố định 0.30.
Giữ policy lọc của bản nguồn; alias mở rộng dùng cho audit/redaction và term-only probe.
`unscreened` giữ `final_caption` trước áp mask, bỏ các cờ từ chối chỉ cho đối chứng; đây không phải
`raw_caption` đầy đủ và không phục hồi được text đã bị xóa khỏi CSV. `redacted` xóa các tên lớp và
alias trong toàn bộ caption. Raw/redacted dùng cùng availability mask; redacted được phép ngắn hơn 5 từ
để phép che từ không bị biến thành phép loại cả caption. Danh sách alias chỉ là heuristic có thể mở rộng trước khi mở test.
So screened với unscreened thay đổi tỷ lệ caption được dùng; cần đọc cùng coverage, không coi chênh lệch đó
là ảnh hưởng thuần của nội dung từ chỉ lớp. Cặp unscreened/redacted đã khớp mask để kiểm tra việc che từ.

Shuffle không dùng nhãn: chỉ hoán vị giữa caption hợp lệ, trong cùng split và cùng nhóm pool/non-pool,
caption candidate chỉ lấy từ pool train. Giữ nguyên mask từng ảnh, lưu ID caption donor. Nhóm chỉ có một
caption hợp lệ không thể derange; trường hợp này được ghi ra manifest, không tự đổi tỷ lệ caption hợp lệ.
Shuffle query trên val/test là đối chứng cố ý phá liên kết ảnh-text giữa các ảnh trong cùng split,
không phải pipeline inference độc lập từng ảnh để triển khai; candidate vẫn chỉ lấy caption từ train pool.

**Cách đọc leak:** DTD dùng chính các thuộc tính texture làm tên lớp. Việc caption sinh từ pixels nói đúng
"striped" không chứng minh model đã đọc nhãn test. Drop sau redaction có thể do mất tín hiệu thị giác hợp lệ.
Caption đảo, text-only probe, mask-only probe, tỷ lệ từ chỉ lớp và intervention giúp phát hiện shortcut;
không phép kiểm riêng lẻ nào chứng minh hoàn toàn không leak. CLIP similarity không phải xác suất leak.

## Cách chạy

1. Gắn DTD, DINOv3 code/ViT-L weights và CLIP ViT-B/32 như notebook nguồn.
2. Đặt `CAPTIONS_INPUT_PATH` đến CSV DTD đã tạo. Nếu CSV cũ còn trong working, notebook tự lấy từ output bản nguồn.
   Nếu chưa có CSV, giữ `CAPTIONS_INPUT_PATH=""`: mặc định tự tạo/resume bằng Florence-2 và CLIP local.
   Model cũ của bạn được tìm trong `/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi/Florence-2-base`.
   Checkpoint Microsoft có `AutoModelForCausalLM` cần Transformers 4.49.x; checkpoint native cần runtime tương thích.
   CSV đầy đủ được dùng lại ngay, không nạp captioner; khi resume CSV thiếu, cấu hình model/prompt phải khớp các dòng đã tạo.
3. Giữ 8 cấu hình mặc định hoặc chốt `ABLATION_NAMES` trước khi train. Mặc định seed 42, tối đa 100 epoch/cấu hình;
   `DTD_ABLATION_SEEDS=42,43,44` cho multi-seed. Toàn bộ suite lâu hơn một lần train thông thường.
4. Chạy lần lượt: caption cache -> audit/CLIP -> probes -> train từng cấu hình -> validation interventions -> final test.
5. Resume từng run từ last.pt, tự bỏ qua run đã xong. Chỉ giữ best.pt sau khi hoàn tất để giảm dung lượng Kaggle.
   Dành chỗ cho nhiều ViT-L checkpoints; có thể chạy một subset nếu giới hạn tài nguyên, nhưng chốt plan trước final test.
   Nhiều seeds làm tăng cả thời gian và dung lượng; có thể cần tách session/output thay vì lưu 24 checkpoint trong Kaggle working.

Kết quả/charts/CSV ở output riêng, không ghi đè notebook hay output nguồn. Không có metric giả trong notebook.
`RUN_FINAL_TEST=False` cho giai đoạn chỉ phát triển trên validation. Khi bật, test chỉ mở sau khi toàn bộ plan đã train xong;
plan và best epoch được khóa trước khi tính test metrics, kết quả được cache để rerun không đánh giá lại.

In [ ]:
# Offline preflight: no pip or network requests are made by this notebook.
import importlib.util
from importlib.metadata import version

_required_modules = {
    "numpy": "numpy", "pandas": "pandas", "torch": "torch", "torchvision": "torchvision",
    "PIL": "Pillow", "matplotlib": "matplotlib", "tqdm": "tqdm", "sklearn": "scikit-learn",
    "transformers": "transformers", "accelerate": "accelerate", "einops": "einops", "timm": "timm",
}
_missing_modules = [package for module, package in _required_modules.items() if importlib.util.find_spec(module) is None]
if _missing_modules:
    raise RuntimeError("Missing offline packages: " + ", ".join(_missing_modules))
_transformers_version = version("transformers")
FLORENCE_BACKEND = "legacy" if _transformers_version.startswith("4.49.") else "native" if _transformers_version.split(".")[0] == "5" else None
if FLORENCE_BACKEND is None:
    print(f"Warning: Transformers {_transformers_version} has not been tested here; use 4.49.x or 5.x")
print("Offline package check passed; transformers", _transformers_version, "| Florence backend:", FLORENCE_BACKEND)

In [ ]:
# === M-TACS: CONFIGURATION ===
import os
import json
import random
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

EXPERIMENT_MODE = "dtd_text_ablation_suite"
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
CLIP_MODEL_LOCAL_PATH = os.getenv("CLIP_MODEL_LOCAL_PATH", "/kaggle/input/datasets/nguyenducxu/vit-32/model_clip_vit")  # Set to the attached local CLIP model folder if no text_embeddings.pt cache is attached.
FLORENCE_MODEL_REF = os.getenv("FLORENCE_MODEL_REF", "florence-community/Florence-2-base" if FLORENCE_BACKEND == "native" else "microsoft/Florence-2-base")  # Complete local checkpoint matching the Transformers backend.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
VLM_MODEL = FLORENCE_MODEL_REF
PROMPT_VERSION = "florence2_detailed_v2"
CAPTION_TASK_PROMPT = "<MORE_DETAILED_CAPTION>"
USE_CACHED_CAPTIONS = True
USE_CACHED_TEXT_EMBEDDINGS = True
ENABLE_LEAKAGE_CHECK = True
DTD_SPLIT = int(os.getenv("DTD_SPLIT", "1"))  # Official DTD split, 1..10.
RUN_ABLATIONS_IN_SEPARATE_RUNS = True
LAMBDA_POLICY = float(globals().get("TACS_POLICY_WEIGHT", 1.0))

CAPTION_MAX_NEW_TOKENS = 48
CAPTION_SEMANTIC_THRESHOLD = 0.30
CAPTION_WORKERS = min(8, os.cpu_count() or 1)
PROMPT_TEXT = "Florence-2 detailed captions; retain 5-8 visual words and screen DTD class terms."
print("Experiment mode:", EXPERIMENT_MODE)
print("Caption model:", VLM_MODEL)
print("Frozen CLIP model:", CLIP_MODEL_NAME)


# Protocol: finalize the plan on validation before looking at final test results.
AB_CONFIG = {
    "visual_only": dict(query_fusion=False, candidate_fusion=False, semantic_selector=False),
    "query_only": dict(query_fusion=True, candidate_fusion=False, semantic_selector=False),
    "candidate_only": dict(query_fusion=False, candidate_fusion=True, semantic_selector=False),
    "both_fusion": dict(query_fusion=True, candidate_fusion=True, semantic_selector=False),
    "full_both": dict(query_fusion=True, candidate_fusion=True, semantic_selector=True),
    "full_shuffle_both": dict(query_fusion=True, candidate_fusion=True, semantic_selector=True,
                              q_transform="shuffle", c_transform="shuffle"),
    "full_unscreened": dict(query_fusion=True, candidate_fusion=True, semantic_selector=True, view="unscreened"),
    "full_redacted": dict(query_fusion=True, candidate_fusion=True, semantic_selector=True, view="redacted"),
    # Optional finer training controls; default fixed-context interventions already test these roles.
    "selector_text_only": dict(query_fusion=False, candidate_fusion=False, semantic_selector=True),
    "full_shuffle_query": dict(query_fusion=True, candidate_fusion=True, semantic_selector=True, q_transform="shuffle"),
    "full_shuffle_candidate": dict(query_fusion=True, candidate_fusion=True, semantic_selector=True, c_transform="shuffle"),
}
DEFAULT_ABLATIONS = ["visual_only", "query_only", "candidate_only", "both_fusion", "full_both",
                     "full_shuffle_both", "full_unscreened", "full_redacted"]
ABLATION_NAMES = [name.strip() for name in os.getenv("DTD_ABLATIONS", ",".join(DEFAULT_ABLATIONS)).split(",") if name.strip()]
TRAIN_SEEDS = [int(seed) for seed in os.getenv("DTD_ABLATION_SEEDS", "42").split(",")]
if not ABLATION_NAMES or len(ABLATION_NAMES) != len(set(ABLATION_NAMES)) or set(ABLATION_NAMES) - set(AB_CONFIG):
    raise ValueError("DTD_ABLATIONS contains duplicate/unknown experiment names")
if not TRAIN_SEEDS or len(TRAIN_SEEDS) != len(set(TRAIN_SEEDS)):
    raise ValueError("Use distinct training seeds")
SELECTED_ABLATIONS = [{"name": name, "view": "screened", "q_transform": "aligned", "c_transform": "aligned", **AB_CONFIG[name]}
                     for name in ABLATION_NAMES]
CAPTIONS_INPUT_PATH = os.getenv("DTD_CAPTIONS_INPUT_PATH", "")  # Set the exact existing DTD captions.csv path here.
GENERATE_CAPTIONS_IF_MISSING = True  # Create/resume the shared CSV automatically; reuse complete captions without loading Florence-2.
RESUME_ABLATIONS = True
KEEP_LAST_CHECKPOINT = False  # Keep best.pt per run; discard only completed runs' redundant resume state.
RUN_VALIDATION_INTERVENTIONS = True
RUN_FINAL_TEST = True  # Final held-out evaluation starts only after all declared training runs finish.
BOOTSTRAP_REPEATS = 200
EXTRA_FORBIDDEN_TERMS_PATH = os.getenv("DTD_EXTRA_FORBIDDEN_TERMS", "")

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True
# Tận dụng Tensor Cores thế hệ mới trên dòng RTX / GPU Ampere+
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

SEED = int(os.getenv("DTD_POOL_SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

# --- ĐƯỜNG DẪN OUTPUT (Ghi vào working của Kaggle) ---
PROJECT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
OUT_PUT_DIR = Path(os.getenv('DTD_ABLATION_OUTPUT', str(PROJECT_DIR / 'outputs' / f'mtacs_dtd_split{DTD_SPLIT}_text_ablation_vitl16')))
OUTPUT_DIR = OUT_PUT_DIR
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

# DTD images and official annotation files.
DATASET_ROOT = Path(os.getenv('DTD_ROOT', '/kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd/dtd'))
if not DATASET_ROOT.is_dir():
    fallback = Path('/kaggle/input/describable-textures-dataset-dtd/dtd')
    if fallback.is_dir():
        DATASET_ROOT = fallback
IMAGE_ROOT = DATASET_ROOT / 'images'
LABEL_ROOT = DATASET_ROOT / 'labels'
if not IMAGE_ROOT.is_dir() or not LABEL_ROOT.is_dir():
    raise FileNotFoundError(
        f'DTD images/ or labels/ not found below {DATASET_ROOT}. Set DTD_ROOT to the attached dtd directory.'
    )
if not 1 <= DTD_SPLIT <= 10:
    raise ValueError('DTD_SPLIT must be in 1..10')

# --- ĐƯỜNG DẪN DINOv3 REPO & WEIGHTS ---
DINOV3_REPO_DIR = os.getenv('DINOV3_REPO_DIR', '/kaggle/input/datasets/nguyenducxu/dinov3-code/dinov3-main')
if not Path(DINOV3_REPO_DIR).exists():
    for p in [Path('/kaggle/input/datasets/sinhnguync/dinov3-code/dinov3-main'), Path('/kaggle/input/dinov3-code/dinov3-main'), Path('/kaggle/input/dinov3-code')]:
        if p.exists():
            DINOV3_REPO_DIR = str(p)
            break

DINOV3_MODEL_NAME = 'dinov3_vitl16'
DINOV3_WEIGHTS = os.getenv('DINOV3_WEIGHTS', '/kaggle/input/datasets/sinhnguync/dinov3-vitl16/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth')
if not Path(DINOV3_WEIGHTS).exists():
    for p in [
        Path('/kaggle/input/datasets/sinhnguync/dinov3-vitl16/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'),
        Path('/kaggle/input/dinov3-vitl16/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'),
        Path('/kaggle/input/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'),
    ]:
        if p.exists():
            DINOV3_WEIGHTS = str(p)
            break

BACKBONE_NAME = DINOV3_MODEL_NAME
FEATURE_DIM = 1024

IMAGE_SIZE = 384
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(DEVICE).total_memory / (1024 ** 3)
    if DEVICE.type == 'cuda' else 0.0
)
if GPU_MEMORY_GB >= 80:
    BATCH_SIZE, RETRIEVAL_BATCH_SIZE, CANDIDATE_FEATURE_BANK_BATCH_SIZE = 32, 128, 64
elif GPU_MEMORY_GB >= 40:
    BATCH_SIZE, RETRIEVAL_BATCH_SIZE, CANDIDATE_FEATURE_BANK_BATCH_SIZE = 16, 64, 32
elif GPU_MEMORY_GB >= 20:
    BATCH_SIZE, RETRIEVAL_BATCH_SIZE, CANDIDATE_FEATURE_BANK_BATCH_SIZE = 8, 32, 16
else:
    BATCH_SIZE, RETRIEVAL_BATCH_SIZE, CANDIDATE_FEATURE_BANK_BATCH_SIZE = 4, 16, 8
print(f'GPU memory: {GPU_MEMORY_GB:.1f} GiB; ViT-L batch sizes: train={BATCH_SIZE}, retrieval={RETRIEVAL_BATCH_SIZE}, feature-refresh={CANDIDATE_FEATURE_BANK_BATCH_SIZE}')
CAPTION_BATCH_SIZE = int(os.getenv('CAPTION_BATCH_SIZE', '64' if GPU_MEMORY_GB >= 40 else '16'))
EPOCHS = int(os.getenv("DTD_ABLATION_EPOCHS", "100"))
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
BACKBONE_LR = 3e-6
HEAD_LR = 1e-4
BACKBONE_WEIGHT_DECAY = 0.05
HEAD_WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
TOKEN_HEAD_DROPOUT = 0.3
FREEZE_BACKBONE_EPOCHS = 0
WARMUP_EPOCHS = 5
PATIENCE = 10
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 0 if os.name == 'nt' else min(8, os.cpu_count() or 0)
NUM_CLASSES = 47  # DTD texture attributes
TOP_K = 5
CONTEXT_TOP_K = 1
RETRIEVAL_SCORE_WEIGHT = 0.0
SUPCON_WEIGHT = 0.05
SUPCON_TEMPERATURE = 0.10
USE_BBOX_CROP = False  # DTD texture classification uses full images.
CANDIDATE_POOL_FRACTION = 0.20
VAL_FRACTION = None  # Official DTD val split is used.
# Set with the VRAM-aware ViT-L batch policy above.
RETRIEVAL_CHUNK_SIZE = 4096
QUERY_CONTEXT_GATE_DROPOUT = 0.20
QUERY_CONTEXT_GATE_BIAS = -0.7
EVAL_TTA_HORIZONTAL_FLIP = False
USE_CONTEXT_LABEL_PRIOR = False
CONTEXT_LABEL_PRIOR_WEIGHT = 0.25
CONTEXT_LABEL_PRIOR_TEMPERATURE = 0.07

TRAIN_ON_FULL_TRAIN = False
FULL_TRAIN_MONITOR_FRACTION = 0.05  # chi dung khi TRAIN_ON_FULL_TRAIN=True

# --- TACS selector hyperparameters ---
TACS_CANDIDATES_PER_QUERY = None  # None nghĩa là dùng toàn bộ candidate pool
TACS_PREFILTER_TOP_SIMILAR = 0
TACS_GUMBEL_TAU = 0.10
TACS_POLICY_WEIGHT = 1.0
TACS_ENTROPY_WEIGHT = 0.01
TACS_SELECTOR_LR = 1e-4
TACS_SELECTOR_WEIGHT_DECAY = 1e-4
TACS_SELECTOR_DROPOUT = 0.10
TACS_SELECTOR_TEMPERATURE = 1.0

# --- Full-pool selector memory controls ---
TACS_USE_FULL_CANDIDATE_POOL = True
CANDIDATE_IMAGE_BANK_DEVICE = 'cpu'  # Preserve GPU memory for ViT-L training
CANDIDATE_IMAGE_BANK_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
# Set with the VRAM-aware ViT-L batch policy above.
FULL_POOL_SCORE_CHUNK_SIZE = 1024

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 48, IMAGE_SIZE + 48)),
    transforms.RandomResizedCrop(
        (IMAGE_SIZE, IMAGE_SIZE),
        scale=(0.60, 1.0),
        ratio=(0.80, 1.25),
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.25,
            contrast=0.25,
            saturation=0.20,
            hue=0.04,
        )
    ], p=0.75),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))
    ], p=0.15),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=0.20,
        scale=(0.02, 0.12),
        ratio=(0.3, 3.3),
        value="random",
    ),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

_mean = globals().get("DINO_MEAN", globals().get("IMAGENET_MEAN", (0.485, 0.456, 0.406)))
_std = globals().get("DINO_STD", globals().get("IMAGENET_STD", (0.229, 0.224, 0.225)))

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=_mean, std=_std),
])


from pathlib import PurePosixPath

class_names = sorted(path.name for path in IMAGE_ROOT.iterdir() if path.is_dir())
if len(class_names) != NUM_CLASSES:
    raise RuntimeError(f'Expected {NUM_CLASSES} DTD texture classes in {IMAGE_ROOT}; found {len(class_names)}')
class_to_id = {name: index for index, name in enumerate(class_names)}

def load_dtd_split(name):
    split_file = LABEL_ROOT / f'{name}{DTD_SPLIT}.txt'
    if not split_file.is_file():
        raise FileNotFoundError(f'Official DTD split file missing: {split_file}')
    records = []
    for raw in split_file.read_text(encoding='utf-8-sig').splitlines():
        rel = raw.strip().replace('\\', '/').lstrip('/')
        if not rel:
            continue
        parts = PurePosixPath(rel).parts
        if parts and parts[0] == 'images':
            parts = parts[1:]
        if len(parts) != 2 or parts[0] not in class_to_id or '..' in parts:
            raise ValueError(f'Unexpected path in {split_file}: {raw!r}')
        rel = PurePosixPath(*parts).as_posix()
        image_path = IMAGE_ROOT.joinpath(*parts)
        if not image_path.is_file():
            raise FileNotFoundError(f'DTD split refers to missing image: {image_path}')
        records.append({
            'image_id': rel, 'rel_path': rel, 'fname': parts[-1],
            'image_path': str(image_path.resolve()), 'class_id': class_to_id[parts[0]],
            'split': name, 'is_train': int(name == 'train'),
            'bbox_x1': 0, 'bbox_y1': 0, 'bbox_x2': 99999, 'bbox_y2': 99999,
        })
    frame = pd.DataFrame.from_records(records)
    if len(frame) != NUM_CLASSES * 40 or frame['image_id'].duplicated().any():
        raise RuntimeError(f'{split_file} should contain 40 distinct images per class; found {len(frame)} rows')
    counts = frame.groupby('class_id').size()
    if len(counts) != NUM_CLASSES or not counts.eq(40).all():
        raise RuntimeError(f'{split_file} has an unexpected per-class distribution')
    frame['row_id'] = np.arange(len(frame), dtype=int)
    return frame

train_df = load_dtd_split('train')
val_df = load_dtd_split('val')
test_df = load_dtd_split('test')
all_ids = pd.concat([train_df['image_id'], val_df['image_id'], test_df['image_id']])
if all_ids.duplicated().any():
    raise RuntimeError('DTD train/val/test split files overlap')
if TRAIN_ON_FULL_TRAIN:
    raise ValueError('TRAIN_ON_FULL_TRAIN is incompatible with the held-out DTD validation split')
full_train_df = train_df.copy()

candidate_rng = np.random.default_rng(SEED)
if CANDIDATE_POOL_FRACTION >= 1.0:
    candidate_pool_positions = np.arange(len(train_df), dtype=int)
else:
    candidate_pool_size = max(1, int(round(len(train_df) * CANDIDATE_POOL_FRACTION)))
    candidate_pool_positions = np.sort(
        candidate_rng.choice(len(train_df), size=candidate_pool_size, replace=False)
    )

candidate_df = train_df.iloc[candidate_pool_positions].reset_index(drop=True)
candidate_df.to_csv(OUT_PUT_DIR / 'candidate_pool.csv', index=False)
val_df.to_csv(OUT_PUT_DIR / 'validation_split.csv', index=False)

print('DATASET_ROOT:', DATASET_ROOT.resolve())
print('IMAGE_ROOT:', IMAGE_ROOT.resolve())
print('OUT_PUT_DIR:', OUT_PUT_DIR.resolve())
print(f'Standard train images: {len(full_train_df):,}')
print(f'Actual train images: {len(train_df):,}')
print(f'Validation images: {len(val_df):,}')
print(f'Train on full train: {TRAIN_ON_FULL_TRAIN}')
print(f'Standard test images: {len(test_df):,}')
print(f'Candidate pool: {len(candidate_df):,}/{len(train_df):,} actual-train images ({CANDIDATE_POOL_FRACTION:.0%})')
print(f'TACS selected context per query: {CONTEXT_TOP_K}')
print(f'TACS uses full candidate pool during training: {TACS_USE_FULL_CANDIDATE_POOL}')
print(f'TACS Gumbel tau: {TACS_GUMBEL_TAU}, policy weight: {TACS_POLICY_WEIGHT}')
print(f'BBox crop enabled: {USE_BBOX_CROP}')
print(f'Train label range: {int(train_df.class_id.min())}..{int(train_df.class_id.max())}')
print(f'Test label range: {int(test_df.class_id.min())}..{int(test_df.class_id.max())}')
display(train_df.head())
display(val_df.head())
display(candidate_df.head())

# image_id is the stable DTD relative path created by load_dtd_split().

## Tái sử dụng caption; giữ nguyên captioner và audit của notebook nguồn

In [ ]:
# === DTD: RESUMABLE FLORENCE-2 CAPTIONS WITH CLIP LEAKAGE AUDIT ===
import csv
import gc
import json
import os
import re
import time
from pathlib import Path, PurePosixPath
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple
import torch
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

TASK_PROMPT = CAPTION_TASK_PROMPT
CSV_COLUMNS = [
    "image_id", "split", "raw_caption", "final_caption", "caption_valid",
    "leakage_detected", "lexical_leakage_detected", "semantic_leakage_detected",
    "semantic_leakage_score", "semantic_leakage_class", "retry_count", "vlm_model", "prompt_version",
]

def normalize_phrase(text: Any) -> str:
    text = str(text).replace("_", " ").replace("/", " ").replace("-", " ").lower()
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def build_forbidden_terms(class_names: Sequence[str], extra_terms: Iterable[str] = ()) -> List[str]:
    terms: Set[str] = set()
    for name in class_names:
        normalized = normalize_phrase(name)
        leaf = normalize_phrase(PurePosixPath(name.replace("\\", "/")).name)
        if normalized:
            terms.add(normalized)
        if leaf:
            terms.add(leaf)
    terms.update(filter(None, (normalize_phrase(term) for term in extra_terms)))
    return sorted(terms, key=lambda value: (-len(value.split()), -len(value), value))


def audit_caption(raw_text: str, forbidden_terms: Sequence[str]) -> Tuple[str, bool, bool, List[str]]:
    words = re.findall(r"[A-Za-z0-9]+(?:['’][A-Za-z0-9]+)?", str(raw_text))
    caption = " ".join(words[:8])
    normalized = f" {normalize_phrase(caption)} "
    matched = [term for term in forbidden_terms if f" {term} " in normalized]
    count = len(normalized.split())
    return caption, 5 <= count <= 8 and not matched, bool(matched), matched


def load_existing(path: Path) -> Dict[str, Dict[str, str]]:
    if not path.is_file() or path.stat().st_size == 0:
        return {}
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        fields = set(reader.fieldnames or [])
        missing = set(CSV_COLUMNS) - fields
        if missing:
            raise ValueError(
                f"Existing CSV predates the semantic audit and lacks {sorted(missing)}. "
                "Choose a new --output path or pass --force to regenerate; do not append new rows to the old schema."
            )
        return {row["image_id"]: row for row in reader if row.get("image_id")}


def append_rows(path: Path, rows: Sequence[Dict[str, Any]]) -> None:
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    header = not path.is_file() or path.stat().st_size == 0
    with path.open("a", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_COLUMNS, extrasaction="ignore")
        if header:
            writer.writeheader()
        writer.writerows(rows)
        handle.flush()
        os.fsync(handle.fileno())


class CaptionDataset(Dataset):
    def __init__(self, records: Sequence[Dict[str, Any]]):
        self.records = list(records)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        row = self.records[index]
        try:
            with Image.open(row["image_path"]) as source:
                image = ImageOps.exif_transpose(source).convert("RGB")
            return row, image, None
        except Exception as exc:
            return row, None, f"{type(exc).__name__}: {exc}"


def collate_images(items):
    return items


def _model_type(path: Path) -> Optional[str]:
    config_path = path / "config.json"
    if not config_path.is_file():
        return None
    try:
        config = json.loads(config_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None
    model_type = str(config.get("model_type", "")).lower()
    architectures = " ".join(config.get("architectures", [])).lower()
    if model_type:
        return model_type
    if "florence2" in architectures or "florence" in architectures:
        return "florence2"
    return None


def resolve_local_model(model_ref: str, expected_type: str, extra_roots: Sequence[Path] = ()) -> Optional[Path]:
    """Find a model folder in Kaggle Inputs, notebook outputs, or HF cache."""
    requested = Path(model_ref).expanduser()
    candidates: List[Path] = []
    if requested.is_dir():
        candidates.append(requested)
        candidates.extend(path.parent for path in requested.rglob("config.json"))

    roots = list(extra_roots)
    roots.extend([
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path.home() / ".cache/huggingface/hub/models--microsoft--Florence-2-base/snapshots",
        Path.home() / ".cache/huggingface/hub/models--florence-community--Florence-2-base/snapshots",
        Path.home() / ".cache/huggingface/hub/models--openai--clip-vit-base-patch32/snapshots",
    ])
    visited: Set[str] = set()
    for root in roots:
        if not root.is_dir():
            continue
        # Kaggle Inputs can contain many images; only inspect model configs to
        # a modest directory depth, which covers attached model/notebook outputs.
        for current, dirs, files in os.walk(root):
            current_path = Path(current)
            try:
                depth = len(current_path.relative_to(root).parts)
            except ValueError:
                dirs[:] = []
                continue
            if depth >= 6:
                dirs[:] = []
            if "config.json" in files:
                key = str(current_path.resolve())
                if key not in visited:
                    visited.add(key)
                    candidates.append(current_path)

    expected = expected_type.lower()
    matching = []
    for path in candidates:
        model_type = _model_type(path)
        if expected in {"florence2_native", "florence2_legacy"}:
            if model_type != "florence2":
                continue
            try:
                config = json.loads((path / "config.json").read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError):
                continue
            legacy = "AutoModelForCausalLM" in config.get("auto_map", {})
            if legacy != (expected == "florence2_legacy"):
                continue
        elif model_type != expected:
            continue
        matching.append(path)
    if not matching:
        return None
    # Prefer an explicit path, then folder names that identify the requested model.
    requested_name = "florence" if expected.startswith("florence2") else "clip"
    matching.sort(key=lambda path: (0 if path == requested else 1, 0 if requested_name in str(path).lower() else 1, len(str(path))))
    return matching[0]


def load_florence(model_id: str, device: torch.device, local_files_only: bool):
    if FLORENCE_BACKEND not in {"legacy", "native"}:
        raise RuntimeError(
            f"Transformers {_transformers_version} is not supported here; use 4.49.x with the Microsoft checkpoint "
            "or 5.x with the florence-community checkpoint."
        )
    expected_format = "florence2_legacy" if FLORENCE_BACKEND == "legacy" else "florence2_native"
    local_model = resolve_local_model(model_id, expected_format, extra_roots=(
        Path("/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi/Florence-2-base"),
    ))
    if local_model is None:
        other_format = "florence2_legacy" if FLORENCE_BACKEND == "native" else "florence2_native"
        other_model = resolve_local_model(model_id, other_format)
        if other_model is not None:
            required_version = "4.49.x" if other_format == "florence2_legacy" else "5.x"
            raise RuntimeError(
                f"Florence-2 weights exist at {other_model}, but their format does not match Transformers {_transformers_version}. "
                f"Use Transformers {required_version} with that checkpoint, or attach a checkpoint matching the current runtime. "
                "After changing Transformers, restart the Kaggle session and Run All. No packages were installed automatically."
            )
        checkpoint = "microsoft/Florence-2-base" if FLORENCE_BACKEND == "legacy" else "florence-community/Florence-2-base"
        raise FileNotFoundError(
            f"No compatible local Florence-2 checkpoint for Transformers {_transformers_version}. "
            f"Attach the complete {checkpoint} folder as a Kaggle Input and set FLORENCE_MODEL_REF to its path. "
            "DTD images and CLIP weights do not contain Florence-2 weights."
        )
    from transformers import AutoProcessor
    dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device.type == "cuda" else torch.float32
    processor = AutoProcessor.from_pretrained(str(local_model), trust_remote_code=FLORENCE_BACKEND == "legacy", local_files_only=True)
    if FLORENCE_BACKEND == "legacy":
        from transformers import AutoModelForCausalLM
        model = AutoModelForCausalLM.from_pretrained(
            str(local_model), torch_dtype=dtype, trust_remote_code=True,
            low_cpu_mem_usage=True, local_files_only=True,
        )
    else:
        from transformers import Florence2ForConditionalGeneration
        model = Florence2ForConditionalGeneration.from_pretrained(
            str(local_model), dtype=dtype, local_files_only=True,
        )
    print(f"Using local Florence-2 ({FLORENCE_BACKEND}): {local_model}")
    return model.to(device).eval(), processor, dtype


def generate_batch(model, processor, images, device, dtype, max_new_tokens: int) -> List[str]:
    inputs = processor(text=[TASK_PROMPT] * len(images), images=images, return_tensors="pt", padding=True)
    moved = {key: value.to(device=device, dtype=dtype if key == "pixel_values" else value.dtype, non_blocking=True)
             for key, value in inputs.items() if torch.is_tensor(value)}
    with torch.inference_mode(), torch.amp.autocast("cuda", dtype=dtype, enabled=device.type == "cuda"):
        generated = model.generate(**moved, max_new_tokens=max_new_tokens, num_beams=1, do_sample=False)
    decoded = processor.batch_decode(generated, skip_special_tokens=False)
    outputs = []
    for text, image in zip(decoded, images):
        parsed = processor.post_process_generation(text, task=TASK_PROMPT, image_size=(image.width, image.height))
        outputs.append(str(parsed.get(TASK_PROMPT, text)).strip() if isinstance(parsed, dict) else str(text).strip())
    return outputs


def load_semantic_auditor(model_path: str, class_names: Sequence[str], device: torch.device, dtype: torch.dtype, local_files_only: bool):
    """Load local CLIP text tower and cache vectors for all DTD class names."""
    from transformers import AutoTokenizer, CLIPModel

    local_model = resolve_local_model(model_path, "clip")
    if local_model is not None:
        model_path = str(local_model)
        print(f"Using local CLIP auditor: {model_path}")
    elif local_files_only:
        raise FileNotFoundError(
            f"Local CLIP model not found for {model_path}. Attach its complete folder as Kaggle Input "
            "or set --clip-model to the correct path. Semantic audit is required by default."
        )
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=local_files_only)
    dtype_arg = {"dtype" if FLORENCE_BACKEND == "native" else "torch_dtype": dtype}
    model = CLIPModel.from_pretrained(model_path, local_files_only=local_files_only, **dtype_arg).to(device).eval()
    labels = [PurePosixPath(name.replace("\\", "/")).name.replace("_", " ").replace("-", " ") for name in class_names]
    prompts = [prompt.format(name=name) for name in labels for prompt in ("a photo of {name}", "a scene showing {name}")]
    vectors = encode_clip_texts(model, tokenizer, prompts, device, dtype, batch_size=128)
    vectors = vectors.reshape(len(labels), 2, -1).mean(dim=1)
    vectors = torch.nn.functional.normalize(vectors, dim=1)
    return model, tokenizer, vectors, labels


@torch.inference_mode()
def encode_clip_texts(model, tokenizer, texts: Sequence[str], device: torch.device, dtype: torch.dtype, batch_size: int) -> torch.Tensor:
    encoded = []
    for start in range(0, len(texts), batch_size):
        tokens = tokenizer(list(texts[start:start + batch_size]), padding=True, truncation=True, return_tensors="pt")
        tokens = {key: value.to(device, non_blocking=True) for key, value in tokens.items()}
        with torch.amp.autocast("cuda", dtype=dtype, enabled=device.type == "cuda"):
            text_output = model.text_model(**tokens, return_dict=True)
            features = model.text_projection(text_output.pooler_output)
        encoded.append(torch.nn.functional.normalize(features.float(), dim=1).cpu())
    return torch.cat(encoded, dim=0) if encoded else torch.empty((0, model.config.projection_dim))


def semantic_audit_captions(
    captions: Sequence[str], model, tokenizer, class_vectors: torch.Tensor, class_labels: Sequence[str],
    device: torch.device, dtype: torch.dtype, threshold: float, batch_size: int,
):
    caption_vectors = encode_clip_texts(model, tokenizer, captions, device, dtype, batch_size).to(device)
    similarities = caption_vectors @ class_vectors.to(device).T
    scores, indices = similarities.max(dim=1)
    return [
        (float(score), str(class_labels[int(index)]), float(score) >= threshold)
        for score, index in zip(scores.cpu(), indices.cpu())
    ]



SHARED_DIR = OUT_PUT_DIR / "shared"
RUNS_DIR = OUT_PUT_DIR / "runs"
SHARED_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
CAPTION_CSV = SHARED_DIR / "captions.csv"
if CAPTIONS_INPUT_PATH:
    requested_csv = Path(CAPTIONS_INPUT_PATH).expanduser()
    if not requested_csv.is_file():
        raise FileNotFoundError(f"DTD_CAPTIONS_INPUT_PATH not found: {requested_csv}")
    if CAPTION_CSV.is_file() and requested_csv.resolve() != CAPTION_CSV.resolve():
        if requested_csv.read_bytes() != CAPTION_CSV.read_bytes():
            raise ValueError("Requested captions differ from this experiment's cached CSV; choose a new output directory")
if not CAPTION_CSV.is_file():
    old_output = PROJECT_DIR / "outputs" / f"mtacs_dtd_split{DTD_SPLIT}_captions_vitl16" / "captions.csv"
    source_csv = Path(CAPTIONS_INPUT_PATH).expanduser() if CAPTIONS_INPUT_PATH else old_output
    if source_csv.is_file():
        shutil.copy2(source_csv, CAPTION_CSV)
        print("Reusing existing DTD captions:", source_csv)
    elif CAPTIONS_INPUT_PATH:
        raise FileNotFoundError(f"DTD_CAPTIONS_INPUT_PATH not found: {source_csv}")
    elif not GENERATE_CAPTIONS_IF_MISSING:
        raise FileNotFoundError(
            "Set CAPTIONS_INPUT_PATH to the existing DTD captions.csv, or enable GENERATE_CAPTIONS_IF_MISSING "
            "with matching local Florence-2 and CLIP assets. No caption improvements are applied here."
        )


caption_expected_ids = {
    str(image_id) for frame in (train_df, val_df, test_df) for image_id in frame["image_id"]
}
caption_cached_rows = load_existing(CAPTION_CSV)
caption_unexpected_ids = set(caption_cached_rows) - caption_expected_ids
if caption_unexpected_ids:
    raise ValueError("Cached captions contain images outside DTD; choose the correct CSV or another output directory")
caption_missing_ids = caption_expected_ids - set(caption_cached_rows)
NEEDS_CAPTION_GENERATION = bool(caption_missing_ids)
if NEEDS_CAPTION_GENERATION and not GENERATE_CAPTIONS_IF_MISSING:
    raise FileNotFoundError(
        f"DTD captions incomplete: {len(caption_cached_rows)}/{len(caption_expected_ids)}. "
        "Enable GENERATE_CAPTIONS_IF_MISSING=True to create/resume captions.csv with the attached local models."
    )
if NEEDS_CAPTION_GENERATION:
    print(f"Creating/resuming DTD captions: cached={len(caption_cached_rows):,}, missing={len(caption_missing_ids):,}")
else:
    print(f"Reusing complete DTD captions: {len(caption_cached_rows):,}; Florence-2 generation skipped.")
print("Caption output:", CAPTION_CSV)

if NEEDS_CAPTION_GENERATION:
    CAPTION_CSV = SHARED_DIR / 'captions.csv'
    caption_records = []
    for split_name, frame in (('train', train_df), ('val', val_df), ('test', test_df)):
        caption_records.extend({
            'image_id': str(row.image_id), 'image_path': str(row.image_path), 'split': split_name,
        } for row in frame[['image_id', 'image_path']].itertuples(index=False))
    if len({row['image_id'] for row in caption_records}) != len(caption_records):
        raise RuntimeError('Duplicate image IDs in DTD caption records')

    existing_captions = load_existing(CAPTION_CSV)
    expected_splits = {row['image_id']: row['split'] for row in caption_records}
    for image_id, old in existing_captions.items():
        if image_id not in expected_splits or old.get('split') != expected_splits[image_id]:
            raise ValueError(f'Existing captions.csv does not match DTD split {DTD_SPLIT}: {image_id}')
        if old.get('prompt_version') != PROMPT_VERSION:
            raise ValueError(f'Existing captions.csv has another prompt version for {image_id}; use a new output directory')
        if old.get('vlm_model') != VLM_MODEL:
            raise ValueError(f'Existing captions.csv uses another Florence model for {image_id}; use a new output directory')
    todo_captions = [row for row in caption_records if row['image_id'] not in existing_captions]
    print(f'DTD captions: cached={len(existing_captions):,}, remaining={len(todo_captions):,}, output={CAPTION_CSV}')

    if todo_captions:
        if DEVICE.type != 'cuda':
            raise RuntimeError('Enable a GPU accelerator before generating all DTD captions and training ViT-L/16')
        clip_asset = resolve_local_model(CLIP_MODEL_LOCAL_PATH, 'clip')
        if clip_asset is None:
            raise FileNotFoundError('A complete local CLIP ViT-B/32 model is required; attach it as Kaggle Input or set CLIP_MODEL_LOCAL_PATH')
        CLIP_MODEL_LOCAL_PATH = str(clip_asset)
        florence_model = florence_processor = clip_auditor = clip_audit_tokenizer = None
        try:
            florence_model, florence_processor, florence_dtype = load_florence(
                FLORENCE_MODEL_REF, DEVICE, local_files_only=True
            )
            clip_auditor, clip_audit_tokenizer, class_vectors, class_labels = load_semantic_auditor(
                CLIP_MODEL_LOCAL_PATH, class_names, DEVICE, florence_dtype, local_files_only=True
            )
            forbidden_terms = build_forbidden_terms(class_names) if ENABLE_LEAKAGE_CHECK else []
            caption_loader = DataLoader(
                CaptionDataset(todo_captions), batch_size=CAPTION_BATCH_SIZE, shuffle=False,
                num_workers=CAPTION_WORKERS, persistent_workers=CAPTION_WORKERS > 0,
                collate_fn=collate_images,
            )
            effective_batch = CAPTION_BATCH_SIZE
            for batch in tqdm(caption_loader, total=len(caption_loader), desc='DTD Florence-2 captions'):
                for row, image, error in batch:
                    if error:
                        raise RuntimeError(f"Cannot read {row['image_id']}: {error}")
                generated = []
                offset = 0
                while offset < len(batch):
                    group = batch[offset:offset + effective_batch]
                    try:
                        generated.extend(generate_batch(
                            florence_model, florence_processor, [image for _, image, _ in group],
                            DEVICE, florence_dtype, CAPTION_MAX_NEW_TOKENS,
                        ))
                        offset += len(group)
                    except torch.cuda.OutOfMemoryError:
                        if effective_batch == 1:
                            raise
                        effective_batch = max(1, effective_batch // 2)
                        torch.cuda.empty_cache()
                        tqdm.write(f'Caption batch reduced to {effective_batch} after CUDA OOM')
                batch_rows = []
                final_texts = []
                for (row, _image, _error), raw in zip(batch, generated):
                    final, lexical_valid, lexical_leak, _matched = audit_caption(raw, forbidden_terms)
                    batch_rows.append({
                        'image_id': row['image_id'], 'split': row['split'], 'raw_caption': raw,
                        'final_caption': final, 'caption_valid': lexical_valid,
                        'leakage_detected': lexical_leak, 'lexical_leakage_detected': lexical_leak,
                        'semantic_leakage_detected': False, 'semantic_leakage_score': 0.0,
                        'semantic_leakage_class': '', 'retry_count': 0,
                        'vlm_model': VLM_MODEL, 'prompt_version': PROMPT_VERSION,
                    })
                    final_texts.append(final)
                semantic_results = semantic_audit_captions(
                    final_texts, clip_auditor, clip_audit_tokenizer, class_vectors, class_labels,
                    DEVICE, florence_dtype, CAPTION_SEMANTIC_THRESHOLD, batch_size=256,
                )
                for row, (score, matched_class, semantic_leak) in zip(batch_rows, semantic_results):
                    row['semantic_leakage_score'] = round(score, 6)
                    row['semantic_leakage_class'] = matched_class
                    row['semantic_leakage_detected'] = semantic_leak
                    row['leakage_detected'] = bool(row['leakage_detected'] or semantic_leak)
                    row['caption_valid'] = bool(row['caption_valid'] and not semantic_leak)
                append_rows(CAPTION_CSV, batch_rows)
        finally:
            del florence_model, florence_processor, clip_auditor, clip_audit_tokenizer
            gc.collect()
            torch.cuda.empty_cache()

    completed_captions = load_existing(CAPTION_CSV)
    if len(completed_captions) != len(caption_records):
        raise RuntimeError(f'Caption CSV is incomplete: {len(completed_captions)}/{len(caption_records)}; rerun this cell to resume')
    print('Complete DTD captions:', len(completed_captions))
    print('Valid captions:', sum(str(row['caption_valid']).lower() in {'true', '1'} for row in completed_captions.values()))

## Caption views, hoán vị không dùng nhãn và CLIP cache riêng

In [ ]:
import copy
import gc
import hashlib
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm


def stable_hash(value):
    import json
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True).encode()).hexdigest()


def write_json(path, value):
    import json
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def read_json(path):
    import json
    return json.loads(Path(path).read_text(encoding="utf-8"))


def seed_ablation(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def metric_scores(labels, predictions):
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, labels=list(range(NUM_CLASSES)),
                                   average="macro", zero_division=0)),
    }


def paired_f1_interval(reference, alternative, repeats=200, seed=123):
    a = reference.set_index("image_id").sort_index()
    b = alternative.set_index("image_id").sort_index()
    if not a.index.equals(b.index) or not np.array_equal(a.ground_truth, b.ground_truth):
        raise ValueError("Paired comparison requires the same image IDs and labels")
    labels = a.ground_truth.to_numpy()
    base, changed = a.prediction.to_numpy(), b.prediction.to_numpy()
    delta = metric_scores(labels, changed)["macro_f1"] - metric_scores(labels, base)["macro_f1"]
    result = {"delta_macro_f1": delta, "changed_prediction_rate": float(np.mean(base != changed)),
              "ci95_low": None, "ci95_high": None}
    if repeats:
        rng = np.random.default_rng(seed)
        groups = [np.flatnonzero(labels == label) for label in np.unique(labels)]
        differences = []
        for _ in range(repeats):
            indices = np.concatenate([rng.choice(group, len(group), replace=True) for group in groups])
            differences.append(metric_scores(labels[indices], changed[indices])["macro_f1"]
                               - metric_scores(labels[indices], base[indices])["macro_f1"])
        result["ci95_low"], result["ci95_high"] = [float(x) for x in np.quantile(differences, [0.025, 0.975])]
    return result
SOURCE_NOTEBOOK_SHA256 = 'fb78dfd1029f890e98f317aff8e9323cd1e3c6dde729fb37f37eef9ff27a8f3d'
HELPER_SOURCE_SHA256 = '09accdb416288c11fda35091e546b72b2f9270a2fb59093ca3202af79e3ff5c3'

In [ ]:
def normalized_text(text):
    import re
    return re.sub(r"[^a-z0-9]+", " ", str(text).replace("_", " ").lower()).strip()


def bool_value(value):
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


DTD_EXTRA_TERMS = {
    "banded": ["bands", "banding"], "blotchy": ["blotches", "blotch"],
    "braided": ["braid", "braids"], "bubbly": ["bubbles", "bubble"],
    "bumpy": ["bumps", "bump"], "chequered": ["checkered", "checkerboard"],
    "cobwebbed": ["cobwebs", "cobweb"], "cracked": ["cracks", "crack"],
    "crosshatched": ["crosshatched", "cross hatching"], "crystalline": ["crystals", "crystal"],
    "dotted": ["dots", "polka dots"], "fibrous": ["fibers", "fibres"],
    "flecked": ["flecks"], "freckled": ["freckles"], "frilly": ["frills"],
    "gauzy": ["gauze"], "gridded": ["grid", "grids"], "grooved": ["grooves"],
    "honeycombed": ["honeycomb"], "interlaced": ["interlacing"],
    "knitted": ["knit", "knitting"], "lacelike": ["lace", "lace like"],
    "lined": ["parallel lines"], "marbled": ["marbling"], "meshed": ["mesh"],
    "paisley": [], "perforated": ["perforations"], "pitted": ["pits"],
    "pleated": ["pleats"], "polka-dotted": ["polka dotted", "polka dots"],
    "porous": ["pores"], "potholed": ["potholes"], "scaly": ["scales"],
    "smeared": ["smears"], "spiralled": ["spiraled", "spirals", "spiral"],
    "sprinkled": ["sprinkles"], "stained": ["stains"], "stratified": ["strata"],
    "striped": ["stripes", "stripe"], "studded": ["studs"],
    "swirly": ["swirls", "swirl"], "veined": ["veins"],
    "waffled": ["waffle"], "woven": ["weave", "weaving"],
    "wrinkled": ["wrinkles"], "zigzagged": ["zigzag", "zig zag"],
}


def caption_term_pattern(names, extra_terms=()):
    import re
    terms = {normalized_text(name) for name in names}
    terms.update(normalized_text(term) for name in names for term in DTD_EXTRA_TERMS.get(name, []))
    terms.update(normalized_text(term) for term in extra_terms)
    terms.discard("")
    ordered = sorted(terms, key=lambda term: (-len(term.split()), -len(term), term))
    if not ordered:
        raise ValueError("No category terms available for the redaction diagnostic")
    expressions = [r"[\W_]+".join(re.escape(word) for word in term.split()) for term in ordered]
    return re.compile(r"(?<!\w)(?:" + "|".join(expressions) + r")(?!\w)", flags=re.IGNORECASE)


def load_ablation_captions(path, split_frames):
    frame = pd.read_csv(path).fillna("")
    required = {"image_id", "final_caption", "caption_valid", "leakage_detected",
                "semantic_leakage_detected", "semantic_leakage_score"}
    if not required.issubset(frame.columns):
        raise ValueError("Caption CSV is missing audit columns: " + str(sorted(required - set(frame.columns))))
    frame["image_id"] = frame.image_id.astype(str)
    if frame.image_id.duplicated().any():
        raise ValueError("Duplicate caption image IDs; resolve the input CSV before this experiment")
    expected = pd.concat([part[["image_id"]].assign(split=split) for split, part in split_frames.items()],
                         ignore_index=True)
    if expected.image_id.duplicated().any():
        raise ValueError("Official train/val/test partitions overlap")
    missing = set(expected.image_id) - set(frame.image_id)
    if missing:
        raise ValueError(f"Caption CSV is incomplete: {len(missing)} missing DTD images")
    # Split membership comes from DTD files, so one image-only CSV can be reused across official splits.
    return expected.merge(frame.drop(columns="split", errors="ignore"), on="image_id", validate="one_to_one")


def prepare_caption_views(frame, names, threshold, extra_terms=()):
    import re
    result = frame.copy()
    pattern = caption_term_pattern(names, extra_terms)
    texts = [str(text).strip() for text in result.final_caption]
    normalized = [normalized_text(text) for text in texts]
    exact = {normalized_text(name) for name in names}
    matched = [pattern.findall(text) for text in normalized]
    redacted = [re.sub(r"\s+", " ", pattern.sub(" ", text)).strip() for text in texts]
    word_count = np.array([len(text.split()) for text in normalized])
    eligible = (word_count >= 5) & (word_count <= 8)
    semantic = pd.to_numeric(result.semantic_leakage_score, errors="coerce").to_numpy(dtype=float)
    lexical = np.array([any(f" {term} " in f" {text} " for term in exact) for text in normalized])
    screened = (eligible & result.caption_valid.map(bool_value).to_numpy()
                & ~result.leakage_detected.map(bool_value).to_numpy() & ~lexical
                & ~result.semantic_leakage_detected.map(bool_value).to_numpy()
                & np.isfinite(semantic) & (semantic < threshold))
    # Match raw/redacted coverage; removing all words must not turn a failed caption into a valid one.
    matched_mask = eligible & np.array([bool(normalized_text(text)) for text in redacted])
    views = {
        "screened": {"texts": texts, "valid": screened},
        "unscreened": {"texts": texts, "valid": matched_mask},
        "redacted": {"texts": redacted, "valid": matched_mask.copy()},
        "term_only": {"texts": [" ".join(terms) for terms in matched],
                      "valid": eligible & np.array([bool(terms) for terms in matched])},
    }
    for view in views.values():
        view["image_ids"] = result.image_id.astype(str).tolist()
    result["normalized_caption"] = normalized
    result["redacted_caption"] = redacted
    result["matched_class_terms"] = [" | ".join(terms) for terms in matched]
    result["exact_class_term_hit"] = lexical
    result["alias_or_class_term_hit"] = [bool(terms) for terms in matched]
    result["screened_valid"] = screened
    result["raw_redacted_matched_valid"] = matched_mask
    result["word_count"] = word_count
    return result, views


def deranged_sources(image_ids, valid, split_by_id, candidate_ids, seed):
    rng = np.random.default_rng(seed)
    candidate_ids = set(candidate_ids)
    sources = np.arange(len(image_ids))
    groups = {}
    for position, image_id in enumerate(image_ids):
        if bool(valid[position]):
            # Never borrow a held-out caption for a train image, or a non-pool caption for a pool image.
            key = (split_by_id[image_id], image_id in candidate_ids)
            groups.setdefault(key, []).append(position)
    singletons = []
    for positions in groups.values():
        if len(positions) < 2:
            singletons.extend(image_ids[i] for i in positions)
            continue
        order = rng.permutation(positions)
        sources[order] = np.roll(order, 1)
    return sources, singletons

def encode_view_cache(views, clip_path, cache_path):
    fingerprint = stable_hash({"schema": 1, "clip_path": str(clip_path),
        "clip_config": hashlib.sha256((Path(clip_path) / "config.json").read_bytes()).hexdigest(),
        "clip_weights": [(path.name, path.stat().st_size, path.stat().st_mtime_ns)
                         for path in sorted(Path(clip_path).iterdir()) if path.suffix in {".safetensors", ".bin"}],
        "views": {key: {"ids": value["image_ids"], "texts": value["texts"],
                        "valid": value["valid"].tolist()} for key, value in views.items()}})
    if Path(cache_path).is_file():
        cached = torch.load(cache_path, map_location="cpu", weights_only=False)
        if cached.get("fingerprint") == fingerprint:
            for name, view in cached["views"].items():
                if not torch.isfinite(view["embeddings"]).all():
                    raise ValueError(f"Non-finite CLIP cache: {name}")
                if torch.count_nonzero(view["embeddings"][~view["valid"]]):
                    raise ValueError(f"Invalid captions have nonzero vectors: {name}")
            return cached
    from transformers import CLIPTextModelWithProjection, CLIPTokenizer
    tokenizer = CLIPTokenizer.from_pretrained(str(clip_path), local_files_only=True)
    encoder = CLIPTextModelWithProjection.from_pretrained(str(clip_path), local_files_only=True).to(DEVICE).eval()
    all_texts = sorted({text for view in views.values() for text, valid in zip(view["texts"], view["valid"]) if valid})
    vectors = {}
    try:
        with torch.inference_mode():
            for offset in tqdm(range(0, len(all_texts), 128), desc="CLIP caption views"):
                texts = all_texts[offset:offset + 128]
                tokens = tokenizer(texts, padding=True, truncation=True,
                    max_length=encoder.config.max_position_embeddings, return_tensors="pt").to(DEVICE)
                embeddings = encoder(**tokens).text_embeds.float().cpu()
                vectors.update(zip(texts, embeddings))
        dimension = int(encoder.config.projection_dim)
        cached_views = {}
        for name, view in views.items():
            matrix = torch.zeros((len(view["image_ids"]), dimension), dtype=torch.float32)
            for i, (text, valid) in enumerate(zip(view["texts"], view["valid"])):
                if valid:
                    matrix[i] = vectors[text]
            cached_views[name] = {**view, "valid": torch.as_tensor(view["valid"], dtype=torch.bool),
                                  "embeddings": matrix}
        cached = {"fingerprint": fingerprint, "embedding_dim": dimension, "views": cached_views}
        temporary = Path(cache_path).with_suffix(".tmp")
        torch.save(cached, temporary)
        temporary.replace(cache_path)
        return cached
    finally:
        del encoder, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def make_ablation_banks(spec, seed, override=None):
    settings = {**spec, **(override or {})}
    view = TEXT_CACHE["views"][settings["view"]]
    mask_view = TEXT_CACHE["views"][settings.get("mask_view", settings["view"])]
    ids = view["image_ids"]
    if mask_view["image_ids"] != ids:
        raise ValueError("Counterfactual masks and caption views must have identical image ordering")
    lookup = {image_id: position for position, image_id in enumerate(ids)}
    candidate_positions = torch.tensor([lookup[image_id] for image_id in CANDIDATE_IDS], dtype=torch.long)
    banks = {"settings": settings, "q_index": lookup, "singletons": {}}
    for role, offset in (("q", 1009), ("c", 2017)):
        indices = np.arange(len(ids))
        if settings.get(role + "_transform", "aligned") == "shuffle":
            indices, singletons = deranged_sources(ids, mask_view["valid"], SPLIT_BY_ID, CANDIDATE_IDS, seed + offset)
            banks["singletons"][role] = singletons
        matrix = view["embeddings"][torch.as_tensor(indices)].clone()
        valid = mask_view["valid"].clone()  # Preserve the recipient's exact availability pattern.
        if settings.get(role + "_transform") == "zero":
            matrix.zero_()
            valid.zero_()
        if role == "c":
            matrix = matrix[candidate_positions].to(DEVICE)
            valid = valid[candidate_positions].to(DEVICE)
            indices = indices[candidate_positions.numpy()]
        banks[role] = {"embeddings": matrix, "valid": valid,
                      "source_ids": [ids[int(i)] for i in indices],
                      "texts": [view["texts"][int(i)] for i in indices]}
    return banks


def batch_caption_inputs(banks, image_ids):
    positions = torch.tensor([banks["q_index"][str(image_id)] for image_id in image_ids], dtype=torch.long)
    return (banks["q"]["embeddings"][positions].to(DEVICE, non_blocking=True),
            banks["q"]["valid"][positions].to(DEVICE, non_blocking=True))

In [ ]:
SPLIT_FRAMES = {"train": train_df, "val": val_df, "test": test_df}
CANDIDATE_IDS = candidate_df.image_id.astype(str).tolist()
CANDIDATE_POSITION = {image_id: i for i, image_id in enumerate(CANDIDATE_IDS)}
caption_df = load_ablation_captions(CAPTION_CSV, SPLIT_FRAMES)
SPLIT_BY_ID = dict(zip(caption_df.image_id, caption_df.split))
extra_terms = []
if EXTRA_FORBIDDEN_TERMS_PATH:
    extra_terms = [line.strip() for line in Path(EXTRA_FORBIDDEN_TERMS_PATH).read_text(encoding="utf-8-sig").splitlines()
                   if line.strip() and not line.lstrip().startswith("#")]
caption_audit, caption_views = prepare_caption_views(caption_df, class_names, CAPTION_SEMANTIC_THRESHOLD, extra_terms)
caption_audit.to_csv(SHARED_DIR / "caption_audit.csv", index=False)
audit_summary = caption_audit.groupby("split").agg(
    count=("image_id", "size"), screened_valid_rate=("screened_valid", "mean"),
    class_term_rate=("exact_class_term_hit", "mean"), alias_term_rate=("alias_or_class_term_hit", "mean"),
    matched_raw_redacted_rate=("raw_redacted_matched_valid", "mean"))
audit_summary.to_csv(SHARED_DIR / "caption_audit_summary.csv")
display(audit_summary)
if not caption_audit.screened_valid.any():
    print("WARNING: screened caption coverage is zero. Screened text ablations will equal visual-only inputs.")
if caption_audit.groupby("split").screened_valid.mean().min() < 0.10:
    print("Low caption coverage: inspect the audit before interpreting a small text effect. Do not tune on test.")
clip_asset = resolve_local_model(CLIP_MODEL_LOCAL_PATH, "clip")
if clip_asset is None:
    raise FileNotFoundError("Attach the complete local CLIP ViT-B/32 folder to encode all caption ablation views")
TEXT_CACHE = encode_view_cache(caption_views, clip_asset, SHARED_DIR / "text_ablation_embeddings.pt")
CLIP_EMBED_DIM = int(TEXT_CACHE["embedding_dim"])
print("Cached caption views:", list(TEXT_CACHE["views"]), "| CLIP dimension:", CLIP_EMBED_DIM)

## Cùng kiến trúc cho mọi cấu hình

Kế thừa backbone/visual fusion từ notebook nguồn; thêm nhánh caption candidate đối xứng với query.
Mọi chế độ dùng cùng classifier head kể cả visual_only; tránh thay head/capacity khi tắt text.
Selector dùng visual score chuẩn hóa và semantic score tùy cấu hình. Lựa chọn rời rạc được học bằng policy gradient
với cùng action đã dùng trong supervised forward; không giả định gradient đi qua index_select.
Reward so sánh query + text query với query + candidate image + các caption tương ứng, dropout tắt ở cả hai nhánh.
Candidate feature bank refresh trước train và sau cập nhật backbone để validation không dùng features cũ.

In [ ]:
from pathlib import Path
import gc
import torch


def resolve_dinov3_weights():
    path = Path(DINOV3_WEIGHTS)
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy file weight: {path}')
    return str(path)


def load_dinov3_model(repo_dir, model_name, weights_path):
    repo_dir = Path(repo_dir)
    if not repo_dir.is_dir():
        raise FileNotFoundError(f'Không tìm thấy thư mục mã nguồn: {repo_dir}')
    weights_path = Path(weights_path)
    if not weights_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy file weight: {weights_path}')
    return torch.hub.load(str(repo_dir), model_name, source='local', weights=str(weights_path))


def load_dinov3_backbone():
    return load_dinov3_model(DINOV3_REPO_DIR, DINOV3_MODEL_NAME, resolve_dinov3_weights())


def ensure_dinov3_assets():
    resolve_dinov3_weights()
    repo_dir = Path(DINOV3_REPO_DIR)
    if not repo_dir.is_dir():
        raise FileNotFoundError(f'Không tìm thấy thư mục mã nguồn: {repo_dir}')
    print('DINOv3 assets đã sẵn sàng từ Kaggle Input.')

class SimpleImageDataset(Dataset):
    def __init__(self, annotations, transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.crop_bbox = bool(crop_bbox)

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        with Image.open(row['image_path']) as source:
            image = source.convert('RGB')
        bbox_columns = {'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2'}
        if self.crop_bbox and bbox_columns.issubset(self.annotations.columns):
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = min(image.width, max(left + 1, int(row['bbox_x2'])))
            bottom = min(image.height, max(top + 1, int(row['bbox_y2'])))
            image = image.crop((left, top, right, bottom))
        return self.transform(image), int(row['class_id']), str(row['fname']), int(row.get('row_id', index))


def extract_cls_features(backbone, images):
    features = backbone(images)
    if isinstance(features, dict):
        if 'x_norm_clstoken' in features:
            return features['x_norm_clstoken']
        if 'x_norm_cls_token' in features:
            return features['x_norm_cls_token']
        if 'x_norm_patchtokens' in features:
            return features['x_norm_patchtokens'].mean(dim=1)
        raise KeyError(f'Khong tim thay CLS token trong output keys: {list(features)}')
    return features[:, 0] if torch.is_tensor(features) and features.ndim == 3 else features


def extract_frozen_dino_embeddings(annotations, backbone, desc):
    dataset = SimpleImageDataset(annotations, eval_transform, crop_bbox=USE_BBOX_CROP)
    loader = DataLoader(dataset, batch_size=RETRIEVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,persistent_workers=True if NUM_WORKERS > 0 else False)
    embeddings = []
    backbone.eval()
    for parameter in backbone.parameters():
        parameter.requires_grad_(False)
    with torch.inference_mode():
        for images, *_ in tqdm(loader, desc=desc, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == 'cuda'):
                features = extract_cls_features(backbone, images)
            features = F.normalize(features.float(), dim=1)
            embeddings.append(features.cpu())
    return torch.cat(embeddings, dim=0)




class DinoV3FrozenSimilarityContextClassifier(nn.Module):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=1024, allow_feature_fallback=False):
        super().__init__()
        self.backbone = load_dinov3_model(repo_dir, model_name, weights_path)
        self.feature_dim = feature_dim
        self.allow_feature_fallback = allow_feature_fallback
        self.active_fusion_mode = 'topk_token_attention'
        self.context_proj = nn.Linear(feature_dim, feature_dim)
        self.query_proj = nn.Linear(feature_dim, feature_dim)
        self.context_attention = nn.Linear(feature_dim, 1)
        self.fusion_norm = nn.LayerNorm(feature_dim)
        self.token_head = nn.Sequential(nn.Dropout(p=TOKEN_HEAD_DROPOUT), nn.Linear(feature_dim, num_classes))
        self.feature_fusion = nn.Sequential(nn.Linear(feature_dim * 2, feature_dim), nn.GELU(), nn.Dropout(p=TOKEN_HEAD_DROPOUT), nn.Linear(feature_dim, num_classes))
        for param in self.backbone.parameters():
            param.requires_grad_(False)

    def _prefix_token_count(self):
        if hasattr(self.backbone, 'num_prefix_tokens'):
            return int(self.backbone.num_prefix_tokens)
        storage_tokens = getattr(self.backbone, 'n_storage_tokens', None)
        if storage_tokens is None:
            storage_tokens = getattr(self.backbone, 'num_register_tokens', 0)
        return 1 + int(storage_tokens or 0)

    def _patch_grid(self, images):
        patch_size = getattr(self.backbone, 'patch_size', 16)
        if isinstance(patch_size, (tuple, list)):
            patch_h, patch_w = int(patch_size[0]), int(patch_size[1])
        else:
            patch_h = patch_w = int(patch_size)
        return images.shape[-2] // patch_h, images.shape[-1] // patch_w

    def _unwrap_tokens(self, prepared):
        if torch.is_tensor(prepared):
            return prepared
        if isinstance(prepared, (tuple, list)):
            for item in prepared:
                if torch.is_tensor(item):
                    return item
        if isinstance(prepared, dict):
            for key in ('x', 'tokens', 'last_hidden_state'):
                if key in prepared and torch.is_tensor(prepared[key]):
                    return prepared[key]
            for item in prepared.values():
                if torch.is_tensor(item):
                    return item
        raise TypeError(f'Cannot unwrap DINO token output of type {type(prepared)}')

    def _prepare_tokens_with_rope(self, images):
        try:
            prepared = self.backbone.prepare_tokens_with_masks(images, None)
        except TypeError:
            prepared = self.backbone.prepare_tokens_with_masks(images)
        tokens = self._unwrap_tokens(prepared)
        rope = None
        if hasattr(self.backbone, 'rope_embed'):
            grid_h, grid_w = self._patch_grid(images)
            try:
                rope = self.backbone.rope_embed(H=grid_h, W=grid_w)
            except TypeError:
                rope = self.backbone.rope_embed(grid_h, grid_w)
        return tokens, rope

    def _repeat_rope_for_query_context(self, rope):
        if rope is None or not torch.is_tensor(rope):
            return rope
        try:
            return rope.repeat_interleave(2, dim=-2)
        except RuntimeError:
            return rope

    def _run_blocks(self, tokens, rope):
        for block in self.backbone.blocks:
            if rope is None:
                tokens = block(tokens)
                continue
            try:
                tokens = block(tokens, rope=rope)
            except TypeError:
                try:
                    tokens = block(tokens, rope)
                except TypeError:
                    tokens = block(tokens)
        if hasattr(self.backbone, 'norm'):
            tokens = self.backbone.norm(tokens)
        return tokens

    def _forward_token_concat_features(self, query_images, context_images):
        query_tokens, query_rope = self._prepare_tokens_with_rope(query_images)
        context_tokens, _ = self._prepare_tokens_with_rope(context_images)
        prefix_tokens = self._prefix_token_count()
        pair_tokens = torch.cat([query_tokens[:, :prefix_tokens], query_tokens[:, prefix_tokens:], context_tokens[:, prefix_tokens:]], dim=1)
        pair_rope = self._repeat_rope_for_query_context(query_rope)
        pair_tokens = self._run_blocks(pair_tokens, pair_rope)
        return pair_tokens[:, 0]

    def _pool_backbone_features(self, images):
        features = self.backbone.forward_features(images)
        if isinstance(features, dict):
            if 'x_norm_clstoken' in features:
                return features['x_norm_clstoken']
            if 'x_norm_patchtokens' in features:
                return features['x_norm_patchtokens'].mean(dim=1)
            for item in features.values():
                if torch.is_tensor(item):
                    return item[:, 0] if item.ndim == 3 else item
        if torch.is_tensor(features):
            return features[:, 0] if features.ndim == 3 else features
        raise TypeError(f'Cannot pool DINO features of type {type(features)}')

    def _forward_feature_fusion(self, query_images, context_images):
        if context_images.ndim == 5:
            context_images = context_images[:, 0]
        query_features = self._pool_backbone_features(query_images)
        context_features = self._pool_backbone_features(context_images)
        self.active_fusion_mode = 'feature_fusion'
        return self.feature_fusion(torch.cat([query_features, context_features], dim=1))

    def _forward_topk_token_attention(self, query_images, context_images, context_scores=None):
        if context_images.ndim == 4:
            context_images = context_images.unsqueeze(1)
        batch_size, num_contexts, channels, height, width = context_images.shape
        query_repeated = query_images.unsqueeze(1).expand(-1, num_contexts, -1, -1, -1).reshape(batch_size * num_contexts, channels, height, width)
        context_flat = context_images.reshape(batch_size * num_contexts, channels, height, width)
        pair_features = self._forward_token_concat_features(query_repeated, context_flat).reshape(batch_size, num_contexts, self.feature_dim)
        query_summary = pair_features.mean(dim=1)
        attention_hidden = torch.tanh(self.context_proj(pair_features) + self.query_proj(query_summary).unsqueeze(1))
        attention_logits = self.context_attention(attention_hidden).squeeze(-1)
        if context_scores is not None:
            if context_scores.ndim == 1:
                context_scores = context_scores.unsqueeze(1)
            attention_logits = attention_logits + RETRIEVAL_SCORE_WEIGHT * context_scores.to(attention_logits.device, dtype=attention_logits.dtype)
        attention_weights = torch.softmax(attention_logits, dim=1)
        fused_features = torch.sum(pair_features * attention_weights.unsqueeze(-1), dim=1)
        fused_features = self.fusion_norm(fused_features)
        self.active_fusion_mode = 'topk_token_attention'
        return fused_features, attention_weights

    def forward(self, query_images, context_images, context_scores=None, return_features=False):
        try:
            fused_features, attention_weights = self._forward_topk_token_attention(query_images, context_images, context_scores=context_scores)
            logits = self.token_head(fused_features)
            if return_features:
                return logits, F.normalize(fused_features.float(), dim=1), attention_weights
            return logits
        except Exception as exc:
            if self.allow_feature_fallback:
                print(f'Top-k token attention failed, fallback to feature fusion: {type(exc).__name__}: {exc}')
                logits = self._forward_feature_fusion(query_images, context_images)
                if return_features:
                    return logits, F.normalize(logits.float(), dim=1), None
                return logits
            raise RuntimeError('Top-k token attention failed. Keep allow_feature_fallback=False for debugging.') from exc

class DinoV3GatedTopKContextClassifier(DinoV3FrozenSimilarityContextClassifier):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=1024, allow_feature_fallback=False):
        super().__init__(
            repo_dir=repo_dir,
            model_name=model_name,
            weights_path=weights_path,
            num_classes=num_classes,
            feature_dim=feature_dim,
            allow_feature_fallback=allow_feature_fallback,
        )
        self.query_norm = nn.LayerNorm(feature_dim)
        self.context_norm = nn.LayerNorm(feature_dim)
        self.final_norm = nn.LayerNorm(feature_dim)

        self.query_head = nn.Sequential(
            nn.Dropout(p=TOKEN_HEAD_DROPOUT),
            nn.Linear(feature_dim, num_classes),
        )

        self.fusion_gate = nn.Sequential(
            nn.Linear(feature_dim * 4, feature_dim),
            nn.GELU(),
            nn.Dropout(p=QUERY_CONTEXT_GATE_DROPOUT),
            nn.Linear(feature_dim, 1),
        )
        nn.init.constant_(self.fusion_gate[-1].bias, QUERY_CONTEXT_GATE_BIAS)
        self.active_fusion_mode = "query_plus_gated_topk_context"

    def forward(self, query_images, context_images, context_scores=None, return_features=False):
        try:
            context_features, attention_weights = self._forward_topk_token_attention(
                query_images,
                context_images,
                context_scores=context_scores,
            )
            query_features = self._pool_backbone_features(query_images)

            query_features = self.query_norm(query_features)
            context_features = self.context_norm(context_features)

            gate_input = torch.cat(
                [
                    query_features,
                    context_features,
                    query_features * context_features,
                    torch.abs(query_features - context_features),
                ],
                dim=1,
            )
            gate = torch.sigmoid(self.fusion_gate(gate_input))

            query_logits = self.query_head(query_features)
            context_logits = self.token_head(context_features)
            logits = query_logits + gate * context_logits

            fused_features = self.final_norm(query_features + gate * context_features)
            self.active_fusion_mode = "query_plus_gated_topk_context"

            if return_features:
                return logits, F.normalize(fused_features.float(), dim=1), attention_weights
            return logits

        except Exception as exc:
            if self.allow_feature_fallback:
                print(f"Gated fusion failed, fallback to parent model: {type(exc).__name__}: {exc}")
                return super().forward(query_images, context_images, context_scores=context_scores, return_features=return_features)
            raise




class TACSSelector(nn.Module):
    def __init__(self, temperature=1.0, normalize=True):
        super().__init__()
        self.temperature = float(temperature)
        self.normalize = bool(normalize)
        self.logit_scale = nn.Parameter(torch.tensor(1.0))

    def score(self, query_features, candidate_features):
        query = query_features.float()
        candidates = candidate_features.float()
        if self.normalize:
            query = F.normalize(query, dim=-1)
            candidates = F.normalize(candidates, dim=-1)
        scores = query @ candidates.T
        return scores * self.logit_scale.clamp(0.01, 100.0) / max(self.temperature, 1e-6)


class DinoV3TACSClassifier(DinoV3GatedTopKContextClassifier):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=1024, allow_feature_fallback=False):
        super().__init__(
            repo_dir=repo_dir,
            model_name=model_name,
            weights_path=weights_path,
            num_classes=num_classes,
            feature_dim=feature_dim,
            allow_feature_fallback=allow_feature_fallback,
        )
        self.selector = TACSSelector(
            temperature=TACS_SELECTOR_TEMPERATURE,
            normalize=True,
        )
        self.active_fusion_mode = 'tacs_full_pool_shared_backbone_selector'

    def selector_encode_images(self, images):
        return self._pool_backbone_features(images)

    def score_full_candidate_pool(self, query_images, candidate_feature_bank):
        query_features = self.selector_encode_images(query_images)
        selector_scores = self.selector.score(query_features, candidate_feature_bank.to(query_features.device))
        return selector_scores, query_features

    def select_full_pool(self, query_images, candidate_feature_bank, excluded_positions=None, use_gumbel=True):
        selector_scores, query_features = self.score_full_candidate_pool(query_images, candidate_feature_bank)
        if excluded_positions is not None:
            positions = excluded_positions.to(selector_scores.device, dtype=torch.long)
            in_pool = positions >= 0
            if in_pool.any():
                selector_scores = selector_scores.clone()
                selector_scores[torch.arange(len(positions), device=selector_scores.device)[in_pool], positions[in_pool]] = -torch.inf
        if self.training and use_gumbel:
            selection_weights = F.gumbel_softmax(
                selector_scores,
                tau=TACS_GUMBEL_TAU,
                hard=True,
                dim=1,
            )
            selected_indices = selection_weights.argmax(dim=1)
        else:
            selected_indices = selector_scores.argmax(dim=1)
            selection_weights = F.one_hot(selected_indices, num_classes=selector_scores.size(1)).to(selector_scores.dtype)
        details = {
            'selector_scores': selector_scores,
            'selection_weights': selection_weights,
            'selected_indices': selected_indices,
            'query_features': query_features,
            'selector_entropy': torch.distributions.Categorical(logits=selector_scores.float()).entropy().mean(),
        }
        self.active_fusion_mode = 'tacs_full_pool_shared_backbone_selector'
        return details

    def forward_selected_context(self, query_images, selected_context_images, return_features=False):
        context_images = selected_context_images.unsqueeze(1)
        return super().forward(query_images, context_images, context_scores=None, return_features=return_features)

    def forward_query_only(self, query_images):
        query_features = self._pool_backbone_features(query_images)
        if hasattr(self, 'query_norm') and hasattr(self, 'query_head'):
            query_features = self.query_norm(query_features)
            return self.query_head(query_features)
        return self.token_head(query_features)




# === M-TACS: TEXT/VISUAL FUSION ===
class MTACSModel(DinoV3TACSClassifier):
    """TACS selector and visual path with a gated CLIP text residual."""
    def __init__(self, *args, clip_dim=512, **kwargs):
        super().__init__(*args, **kwargs)
        self.text_projection = nn.Sequential(
            nn.Linear(clip_dim, FEATURE_DIM),
            nn.LayerNorm(FEATURE_DIM),
            nn.Dropout(0.10),
        )
        self.text_gate = nn.Sequential(
            nn.Linear(FEATURE_DIM * 2, FEATURE_DIM),
            nn.GELU(),
            nn.Linear(FEATURE_DIM, 1),
        )
        self.mtacs_norm = nn.LayerNorm(FEATURE_DIM)
        self.mtacs_classifier = nn.Sequential(
            nn.Dropout(TOKEN_HEAD_DROPOUT),
            nn.Linear(FEATURE_DIM, NUM_CLASSES),
        )
        nn.init.constant_(self.text_gate[-1].bias, -1.0)

    def _fuse_text(self, visual_features, text_embeddings, text_valid_mask):
        text_features = self.text_projection(text_embeddings.float())
        gate_input = torch.cat([visual_features.float(), text_features], dim=1)
        gate = torch.sigmoid(self.text_gate(gate_input))
        valid = text_valid_mask.to(gate.device, dtype=gate.dtype).view(-1, 1)
        gate = gate * valid
        fused = self.mtacs_norm(visual_features.float() + gate * text_features)
        return self.mtacs_classifier(fused), fused, gate.squeeze(1)

    def forward_selected_context_with_text(
        self, query_images, selected_context_images, text_embeddings, text_valid_mask,
        return_features=False,
    ):
        _, visual_features, attention = super().forward_selected_context(
            query_images, selected_context_images, return_features=True
        )
        logits, fused_features, gate = self._fuse_text(
            visual_features, text_embeddings, text_valid_mask
        )
        if return_features:
            return logits, F.normalize(fused_features.float(), dim=1), attention, gate
        return logits, gate

    def forward_query_with_text(
        self, query_images, text_embeddings, text_valid_mask, return_features=False
    ):
        query_features = self.query_norm(self._pool_backbone_features(query_images))
        logits, fused_features, gate = self._fuse_text(
            query_features, text_embeddings, text_valid_mask
        )
        if return_features:
            return logits, F.normalize(fused_features.float(), dim=1), gate
        return logits, gate


def build_candidate_image_bank(annotations, transform, device, dtype):
    dataset = SimpleImageDataset(annotations, transform, crop_bbox=USE_BBOX_CROP)
    loader = DataLoader(dataset, batch_size=RETRIEVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    images_bank = []
    labels = []
    fnames = []
    for images, labels_batch, fnames_batch, *_ in tqdm(loader, desc='build candidate image bank', leave=False):
        images_bank.append(images.to(device=device, dtype=dtype, non_blocking=True).contiguous())
        labels.extend([int(x) for x in labels_batch.tolist()])
        fnames.extend(fnames_batch)
    return torch.cat(images_bank, dim=0), torch.tensor(labels, dtype=torch.long, device=device), fnames


@torch.no_grad()
def refresh_candidate_feature_bank(network, candidate_image_bank, batch_size=CANDIDATE_FEATURE_BANK_BATCH_SIZE):
    was_training = network.training
    network.eval()
    features = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(candidate_image_bank), batch_size), desc='refresh full candidate feature bank', leave=False):
            end = min(start + batch_size, len(candidate_image_bank))
            images = candidate_image_bank[start:end].to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == 'cuda'):
                feats = network.selector_encode_images(images)
            features.append(feats.float().detach().cpu())
    if was_training:
        network.train()
    return torch.cat(features, dim=0)

In [ ]:
def make_ablation_model_class(base_class, visual_class):
    class DTDAblationModel(base_class):
        """All configurations share this class, parameter count, and classifier head."""
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.context_text_projection = copy.deepcopy(self.text_projection)
            self.context_text_gate = copy.deepcopy(self.text_gate)
            self.selector.semantic_mix_logit = nn.Parameter(torch.tensor(0.0))

        def select_context(self, query_images, feature_bank, q_text, q_valid, banks,
                           excluded_positions, semantic=False, sample=False):
            scores, _ = self.score_full_candidate_pool(query_images, feature_bank)
            scores = scores.float()
            scores = (scores - scores.mean(1, keepdim=True)) / scores.std(1, keepdim=True, unbiased=False).clamp_min(1e-6)
            if semantic:
                text_scores = F.normalize(q_text.float(), dim=1) @ F.normalize(banks["c"]["embeddings"].float(), dim=1).T
                valid = q_valid[:, None] & banks["c"]["valid"][None, :]
                count = valid.sum(1, keepdim=True).clamp_min(1)
                mean = (text_scores * valid).sum(1, keepdim=True) / count
                variance = ((text_scores - mean).square() * valid).sum(1, keepdim=True) / count
                text_scores = (text_scores - mean) / variance.sqrt().clamp_min(1e-6) * valid
                scores = scores + torch.sigmoid(self.selector.semantic_mix_logit) * text_scores
            positions = excluded_positions.to(scores.device)
            in_pool = positions >= 0
            if in_pool.any():
                scores = scores.clone()
                scores[torch.arange(len(scores), device=scores.device)[in_pool], positions[in_pool]] = -torch.inf
            if not torch.isfinite(scores).any(1).all():
                raise ValueError("No candidate remains after self-exclusion")
            distribution = torch.distributions.Categorical(logits=scores)
            indices = distribution.sample() if sample else scores.argmax(1)
            return {"indices": indices, "log_prob": distribution.log_prob(indices),
                    "entropy": distribution.entropy().mean()}

        def forward_ablated(self, query_images, context_images, q_text, q_valid, c_text, c_valid, settings):
            if context_images is None:
                visual = self.query_norm(self._pool_backbone_features(query_images))
            else:
                _, visual, _ = visual_class.forward_selected_context(self, query_images, context_images, return_features=True)
            q_features = self.text_projection(q_text.float())
            c_features = self.context_text_projection(c_text.float())
            q_gate = torch.sigmoid(self.text_gate(torch.cat([visual.float(), q_features], 1))).squeeze(1)
            c_gate = torch.sigmoid(self.context_text_gate(torch.cat([visual.float(), c_features], 1))).squeeze(1)
            q_gate = q_gate * q_valid.to(q_gate.dtype) * bool(settings["query_fusion"])
            c_gate = c_gate * c_valid.to(c_gate.dtype) * bool(settings["candidate_fusion"])
            fused = self.mtacs_norm(visual.float() + q_gate[:, None] * q_features + c_gate[:, None] * c_features)
            return {"logits": self.mtacs_classifier(fused), "features": F.normalize(fused, dim=1),
                    "query_gate": q_gate, "candidate_gate": c_gate}
    return DTDAblationModel


class DTDImageQueryDataset(Dataset):
    def __init__(self, annotations, transform):
        self.rows = annotations.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, position):
        from PIL import Image
        row = self.rows.iloc[position]
        with Image.open(row.image_path) as source:
            image = source.convert("RGB")
        return {"query_image": self.transform(image), "label": int(row.class_id), "image_id": str(row.image_id)}


def seed_worker(_):
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def make_ablation_loaders(seed):
    common = {"batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
              "pin_memory": DEVICE.type == "cuda", "worker_init_fn": seed_worker}
    return {
        split: DataLoader(DTDImageQueryDataset(frame, train_transform if split == "train" else eval_transform),
                          shuffle=split == "train", generator=torch.Generator().manual_seed(seed), **common)
        for split, frame in SPLIT_FRAMES.items()
    }


def candidate_images_at(indices):
    return candidate_image_bank.index_select(0, indices.to(candidate_image_bank.device)).to(DEVICE, non_blocking=True)


def run_forward(network, batch, banks, feature_bank, training=False, fixed_positions=None):
    settings = banks["settings"]
    query = batch["query_image"].to(DEVICE, non_blocking=True)
    q_text, q_valid = batch_caption_inputs(banks, batch["image_id"])
    excluded = torch.tensor([CANDIDATE_POSITION.get(str(i), -1) for i in batch["image_id"]], device=DEVICE)
    if fixed_positions is None:
        details = network.select_context(query, feature_bank, q_text, q_valid, banks, excluded,
                                         semantic=settings["semantic_selector"], sample=training)
        indices = details["indices"]
    else:
        indices = torch.tensor([fixed_positions[str(i)] for i in batch["image_id"]], device=DEVICE)
        if ((indices < 0) | (indices >= len(CANDIDATE_IDS))).any() or (indices == excluded).any():
            raise ValueError("Invalid fixed candidate positions")
        details = None
    context = candidate_images_at(indices)
    c_text, c_valid = banks["c"]["embeddings"][indices], banks["c"]["valid"][indices]
    output = network.forward_ablated(query, context, q_text, q_valid, c_text, c_valid, settings)
    return output, details, indices, (query, q_text, q_valid, c_text, c_valid, context)

In [ ]:
def ablation_supcon(features, labels):
    logits = F.normalize(features.float(), dim=1) @ F.normalize(features.float(), dim=1).T / SUPCON_TEMPERATURE
    diagonal = torch.eye(len(labels), device=labels.device, dtype=torch.bool)
    positive = (labels[:, None] == labels[None, :]) & ~diagonal
    valid = positive.sum(1) > 0
    if not valid.any():
        return features.sum() * 0.0
    log_prob = logits - torch.logsumexp(logits.masked_fill(diagonal, -torch.inf), 1, keepdim=True)
    mean = log_prob.masked_fill(~positive, 0).sum(1) / positive.sum(1).clamp_min(1)
    return -mean[valid].mean()


def train_ablation_epoch(network, loader, banks, feature_bank, optimizer, scaler, epoch):
    network.train()
    for parameter in network.backbone.parameters():
        parameter.requires_grad_(epoch > FREEZE_BACKBONE_EPOCHS)
    labels_all, predictions = [], []
    totals = {key: 0.0 for key in ("loss", "reward", "policy_loss", "query_gate", "candidate_gate")}
    count = 0
    for batch in tqdm(loader, desc=f"train epoch {epoch}", leave=False):
        labels = batch["label"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            output, details, _, inputs = run_forward(network, batch, banks, feature_bank, training=True)
            ce = F.cross_entropy(output["logits"].float(), labels, label_smoothing=LABEL_SMOOTHING)
            supcon = ablation_supcon(output["features"], labels)
            query, q_text, q_valid, c_text, c_valid, context = inputs
            # Evaluate both reward branches without dropout; candidate image/text vary together.
            network.eval()
            with torch.no_grad():
                baseline = network.forward_ablated(query, None, q_text, q_valid, c_text,
                                                  torch.zeros_like(c_valid), banks["settings"])["logits"]
                contextual = network.forward_ablated(query, context, q_text, q_valid, c_text,
                                                    c_valid, banks["settings"])["logits"]
                rewards = (F.cross_entropy(baseline.float(), labels, reduction="none")
                           - F.cross_entropy(contextual.float(), labels, reduction="none"))
                advantages = ((rewards - rewards.mean()) / rewards.std(unbiased=False).clamp_min(1e-6)
                              if len(rewards) > 1 else rewards)
            network.train()
            policy = -(details["log_prob"] * advantages.detach()).mean()
            loss = ce + SUPCON_WEIGHT * supcon + TACS_POLICY_WEIGHT * policy - TACS_ENTROPY_WEIGHT * details["entropy"]
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss; stop rather than write an invalid checkpoint")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(network.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        n = len(labels)
        count += n
        labels_all.extend(labels.cpu().tolist())
        predictions.extend(output["logits"].detach().argmax(1).cpu().tolist())
        for key, value in (("loss", loss), ("policy_loss", policy), ("reward", rewards.mean()),
                           ("query_gate", output["query_gate"].mean()),
                           ("candidate_gate", output["candidate_gate"].mean())):
            totals[key] += float(value.detach()) * n
    return {**metric_scores(labels_all, predictions), **{key: value / count for key, value in totals.items()}}


@torch.inference_mode()
def evaluate_ablation(network, loader, banks, feature_bank, fixed_positions=None):
    network.eval()
    rows, probabilities = [], []
    loss_sum = 0.0
    for batch in tqdm(loader, desc="evaluate", leave=False):
        labels = batch["label"].to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            output, _, indices, inputs = run_forward(network, batch, banks, feature_bank, fixed_positions=fixed_positions)
        probs = output["logits"].float().softmax(1).cpu()
        probabilities.append(probs)
        loss_sum += float(F.cross_entropy(output["logits"].float(), labels, reduction="sum",
                                          label_smoothing=LABEL_SMOOTHING))
        for j, image_id in enumerate(batch["image_id"]):
            q_position, c_position = banks["q_index"][str(image_id)], int(indices[j])
            prediction, truth = int(probs[j].argmax()), int(labels[j])
            rows.append({"image_id": str(image_id), "ground_truth": truth, "prediction": prediction,
                "correct": prediction == truth, "confidence": float(probs[j].max()),
                "selected_context_position": c_position, "selected_context_id": CANDIDATE_IDS[c_position],
                "query_caption_source_id": banks["q"]["source_ids"][q_position],
                "candidate_caption_source_id": banks["c"]["source_ids"][c_position],
                "query_caption": banks["q"]["texts"][q_position],
                "candidate_caption": banks["c"]["texts"][c_position],
                "query_caption_valid": bool(inputs[2][j]), "candidate_caption_valid": bool(inputs[4][j]),
                "query_gate": float(output["query_gate"][j]), "candidate_gate": float(output["candidate_gate"][j])})
    frame = pd.DataFrame(rows)
    metrics = {**metric_scores(frame.ground_truth, frame.prediction), "loss": loss_sum / len(frame),
        "query_caption_rate": float(frame.query_caption_valid.mean()),
        "candidate_caption_rate": float(frame.candidate_caption_valid.mean()),
        "mean_query_gate": float(frame.query_gate.mean()), "mean_candidate_gate": float(frame.candidate_gate.mean())}
    return metrics, frame, torch.cat(probabilities)


def create_ablation_model(seed):
    seed_ablation(seed)
    return AblationModel(DINOV3_REPO_DIR, DINOV3_MODEL_NAME, resolve_dinov3_weights(), NUM_CLASSES,
                         FEATURE_DIM, allow_feature_fallback=False, clip_dim=CLIP_EMBED_DIM).to(DEVICE)


def optimizer_for(network):
    groups = []
    for prefix, learning_rate, decay in (("backbone", BACKBONE_LR, BACKBONE_WEIGHT_DECAY),
                                          ("selector", TACS_SELECTOR_LR, TACS_SELECTOR_WEIGHT_DECAY),
                                          ("heads", HEAD_LR, HEAD_WEIGHT_DECAY)):
        parameters = [p for name, p in network.named_parameters()
                      if (name.startswith(prefix + ".") if prefix != "heads"
                          else not name.startswith(("backbone.", "selector.")))]
        groups.append({"params": parameters, "lr": learning_rate, "weight_decay": decay})
    return torch.optim.AdamW(groups)


def refresh_ablation_bank(network):
    return refresh_candidate_feature_bank(network, candidate_image_bank).clone().to(DEVICE)


def run_identity(spec, seed):
    return {"protocol": PROTOCOL_FINGERPRINT, "spec": spec, "training_seed": seed}


def checkpoint_cpu(network, optimizer, scheduler, scaler, history, best_loss, best_epoch, bad_epochs, identity, loaders):
    return {"identity": identity, "epoch": history[-1]["epoch"], "history": history,
            "model": {key: value.detach().cpu() for key, value in network.state_dict().items()},
            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
            "best_loss": best_loss, "best_epoch": best_epoch, "bad_epochs": bad_epochs,
            "python_rng": random.getstate(), "numpy_rng": np.random.get_state(), "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            "train_loader_rng": loaders["train"].generator.get_state()}


def atomic_torch_save(value, path):
    temporary = Path(path).with_suffix(".tmp")
    torch.save(value, temporary)
    temporary.replace(path)


def train_one_configuration(spec, seed):
    run_dir = RUNS_DIR / spec["name"] / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    identity = run_identity(spec, seed)
    config_path, finished = run_dir / "config.json", run_dir / "completed.json"
    if config_path.is_file() and read_json(config_path) != identity:
        raise ValueError(f"Changed experiment at {run_dir}; choose another output directory")
    write_json(config_path, identity)
    if finished.is_file():
        print("Already trained:", spec["name"], seed)
        return run_dir
    network = create_ablation_model(seed)
    loaders, banks = make_ablation_loaders(seed), make_ablation_banks(spec, seed)
    write_json(run_dir / "caption_assignment_manifest.json", {
        "view": spec["view"], "singletons_not_deranged": banks["singletons"],
        "query_transform": spec["q_transform"], "candidate_transform": spec["c_transform"],
        "query_fusion": spec["query_fusion"], "candidate_fusion": spec["candidate_fusion"],
        "semantic_selector": spec["semantic_selector"],
    })
    assignments = []
    for role, ids in (("q", TEXT_CACHE["views"][spec["view"]]["image_ids"]), ("c", CANDIDATE_IDS)):
        assignments.extend({"role": role, "image_id": image_id, "caption_source_id": source_id,
                            "caption_available": bool(valid)}
                           for image_id, source_id, valid in zip(ids, banks[role]["source_ids"], banks[role]["valid"].cpu()))
    pd.DataFrame(assignments).to_csv(run_dir / "caption_source_assignments.csv", index=False)
    optimizer = optimizer_for(network)
    def learning_rate_factor(epoch):
        if epoch < WARMUP_EPOCHS:
            return 0.2 + 0.8 * epoch / max(WARMUP_EPOCHS, 1)
        return max(0.001, 0.5 * (1 + math.cos(math.pi * (epoch - WARMUP_EPOCHS)
                                             / max(EPOCHS - WARMUP_EPOCHS, 1))))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, learning_rate_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    last_path, best_path = run_dir / "last.pt", run_dir / "best.pt"
    history, best_loss, best_epoch, bad_epochs, start_epoch = [], float("inf"), 0, 0, 1
    if RESUME_ABLATIONS and last_path.is_file():
        state = torch.load(last_path, map_location="cpu", weights_only=False)
        if state["identity"] != identity:
            raise ValueError("Resume checkpoint belongs to another experiment")
        network.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        history, best_loss, best_epoch, bad_epochs = state["history"], state["best_loss"], state["best_epoch"], state["bad_epochs"]
        start_epoch = state["epoch"] + 1
        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])
        torch.set_rng_state(state["torch_rng"])
        if state["cuda_rng"]:
            torch.cuda.set_rng_state_all(state["cuda_rng"])
        loaders["train"].generator.set_state(state["train_loader_rng"])
        del state
    elif last_path.is_file() or best_path.is_file():
        raise ValueError(f"Unfinished existing run: {run_dir}; enable RESUME_ABLATIONS or choose a new directory")
    try:
        for epoch in range(start_epoch, EPOCHS + 1):
            if bad_epochs >= PATIENCE:
                break
            bank = refresh_ablation_bank(network)
            training = train_ablation_epoch(network, loaders["train"], banks, bank, optimizer, scaler, epoch)
            del bank
            # The backbone changed during training; validation must use its current candidate features.
            bank = refresh_ablation_bank(network)
            validation, _, _ = evaluate_ablation(network, loaders["val"], banks, bank)
            del bank
            scheduler.step()
            improved = validation["loss"] < best_loss - 1e-6
            if improved:
                best_loss, best_epoch, bad_epochs = validation["loss"], epoch, 0
            else:
                bad_epochs += 1
            history.append({"epoch": epoch, **{"train_" + key: value for key, value in training.items()},
                            **{"val_" + key: value for key, value in validation.items()}})
            if improved:
                atomic_torch_save({"identity": identity, "epoch": epoch, "validation": validation,
                    "class_names": class_names,
                    "model": {key: value.detach().cpu() for key, value in network.state_dict().items()}}, best_path)
            atomic_torch_save(checkpoint_cpu(network, optimizer, scheduler, scaler, history, best_loss,
                                            best_epoch, bad_epochs, identity, loaders), last_path)
            pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
            print(f'{spec["name"]} seed={seed} epoch={epoch} val_f1={validation["macro_f1"]:.4f} '
                  f'val_loss={validation["loss"]:.4f} q_gate={validation["mean_query_gate"]:.3f} '
                  f'c_gate={validation["mean_candidate_gate"]:.3f}')
        best = torch.load(best_path, map_location="cpu", weights_only=False)
        network.load_state_dict(best["model"])
        del best
        bank = refresh_ablation_bank(network)
        validation, predictions, _ = evaluate_ablation(network, loaders["val"], banks, bank)
        predictions.to_csv(run_dir / "val_predictions.csv", index=False)
        write_json(run_dir / "val_metrics.json", validation)
        write_json(finished, {"identity": identity, "best_epoch": best_epoch, "best_val_loss": best_loss})
        if not KEEP_LAST_CHECKPOINT and last_path.is_file():
            last_path.unlink()  # Only this run's redundant resume file; keep its best weights.
        return run_dir
    finally:
        del network, optimizer, scheduler, scaler, loaders, banks
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
def caption_label_diagnostics(split="val"):
    if split == "test" and not (OUT_PUT_DIR / "locked_test_plan.json").is_file():
        raise ValueError("Lock the complete evaluation plan before computing test label-word diagnostics")
    frame = SPLIT_FRAMES[split][["image_id", "class_id"]].merge(caption_audit, on="image_id", validate="one_to_one")
    exact_hits, alias_hits = [], []
    for row in frame.itertuples(index=False):
        label = class_names[int(row.class_id)]
        exact_hits.append(f" {normalized_text(label)} " in f" {row.normalized_caption} ")
        alias_hits.append(bool(caption_term_pattern([label]).search(row.normalized_caption)))
    frame["true_class_name_in_caption"] = exact_hits
    frame["true_class_name_or_alias_in_caption"] = alias_hits
    columns = ["class_id", "image_id", "true_class_name_in_caption", "true_class_name_or_alias_in_caption", "screened_valid"]
    frame[columns].to_csv(OUT_PUT_DIR / f"caption_label_diagnostics_{split}.csv", index=False)
    result = {"split": split, "n": len(frame), "true_class_name_rate": float(np.mean(exact_hits)),
              "true_class_or_alias_rate": float(np.mean(alias_hits))}
    write_json(OUT_PUT_DIR / f"caption_label_summary_{split}.json", result)
    return result


def caption_probes():
    rows = []
    view = TEXT_CACHE["views"]["unscreened"]
    position = {image_id: i for i, image_id in enumerate(view["image_ids"])}
    train_pos = [position[str(i)] for i in train_df.image_id]
    val_pos = [position[str(i)] for i in val_df.image_id]
    mask_features = np.column_stack([TEXT_CACHE["views"][key]["valid"].numpy().astype(float)
                                    for key in ("screened", "unscreened", "term_only")])
    matrices = {key: item["embeddings"].numpy() for key, item in TEXT_CACHE["views"].items()}
    matrices["availability_mask_only"] = mask_features
    for name, matrix in matrices.items():
        probe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, random_state=SEED))
        probe.fit(matrix[train_pos], train_df.class_id.to_numpy())
        prediction = probe.predict(matrix[val_pos])
        metrics = metric_scores(val_df.class_id.to_numpy(), prediction)
        rows.append({"probe": name, **metrics, "split": "val", "n": len(val_pos)})
    result = pd.DataFrame(rows)
    result.to_csv(OUT_PUT_DIR / "caption_probes_val.csv", index=False)
    return result


def validation_interventions(spec, seed):
    run_dir = RUNS_DIR / spec["name"] / f"seed_{seed}"
    network = create_ablation_model(seed)
    best = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)
    if best["identity"] != run_identity(spec, seed):
        raise ValueError("Wrong checkpoint for validation interventions")
    network.load_state_dict(best["model"])
    del best
    loader = make_ablation_loaders(seed)["val"]
    bank = refresh_ablation_bank(network)
    base_banks = make_ablation_banks(spec, seed)
    _, base, _ = evaluate_ablation(network, loader, base_banks, bank)
    fixed = dict(zip(base.image_id, base.selected_context_position))
    interventions = {
        "zero_query_fixed_context": {"q_transform": "zero"},
        "zero_candidate_fixed_context": {"c_transform": "zero"},
        "zero_both_fixed_context": {"q_transform": "zero", "c_transform": "zero"},
        "shuffle_query_fixed_context": {"q_transform": "shuffle"},
        "shuffle_candidate_fixed_context": {"c_transform": "shuffle"},
        "shuffle_both_fixed_context": {"q_transform": "shuffle", "c_transform": "shuffle"},
        "redact_both_fixed_context": {"view": "redacted", "mask_view": spec["view"]},
        "zero_both_reselect": {"q_transform": "zero", "c_transform": "zero"},
        "shuffle_both_reselect": {"q_transform": "shuffle", "c_transform": "shuffle"},
    }
    rows = []
    try:
        for name, override in interventions.items():
            banks = make_ablation_banks(spec, seed, override)
            metrics, predictions, _ = evaluate_ablation(network, loader, banks, bank,
                                                        fixed_positions=fixed if "fixed_context" in name else None)
            paired = paired_f1_interval(base, predictions, BOOTSTRAP_REPEATS, seed)
            changed_context = float(np.mean(base.selected_context_id.to_numpy() != predictions.selected_context_id.to_numpy()))
            rows.append({"experiment": spec["name"], "seed": seed, "intervention": name,
                         **metrics, **paired, "changed_context_rate": changed_context})
            predictions.to_csv(run_dir / f"val_{name}.csv", index=False)
        result = pd.DataFrame(rows)
        result.to_csv(run_dir / "val_interventions.csv", index=False)
        return result
    finally:
        del network, bank, loader, base_banks
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def evaluate_locked_test():
    plan = {"protocol": PROTOCOL_FINGERPRINT, "runs": []}
    for spec in SELECTED_ABLATIONS:
        for seed in TRAIN_SEEDS:
            run_dir = RUNS_DIR / spec["name"] / f"seed_{seed}"
            if not (run_dir / "completed.json").is_file():
                raise RuntimeError("Complete every declared training run before opening test")
            completed = read_json(run_dir / "completed.json")
            if completed["identity"] != run_identity(spec, seed):
                raise ValueError("Completed run no longer matches the locked protocol")
            plan["runs"].append({"experiment": spec["name"], "seed": seed,
                                 "best_epoch": completed["best_epoch"]})
    lock = OUT_PUT_DIR / "locked_test_plan.json"
    if lock.is_file() and read_json(lock) != plan:
        raise ValueError("Test plan changed after evaluation; use a separate experiment directory")
    write_json(lock, plan)
    caption_label_diagnostics("test")
    rows = []
    for spec in SELECTED_ABLATIONS:
        for seed in TRAIN_SEEDS:
            run_dir = RUNS_DIR / spec["name"] / f"seed_{seed}"
            target = run_dir / "test_metrics.json"
            if target.is_file():
                rows.append({"experiment": spec["name"], "seed": seed, **read_json(target)})
                continue
            network = create_ablation_model(seed)
            state = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)
            if state["identity"] != run_identity(spec, seed):
                raise ValueError("Wrong best checkpoint for test")
            network.load_state_dict(state["model"])
            del state
            bank = loader = banks = None
            try:
                bank = refresh_ablation_bank(network)
                loader, banks = make_ablation_loaders(seed)["test"], make_ablation_banks(spec, seed)
                metrics, predictions, _ = evaluate_ablation(network, loader, banks, bank)
                predictions.to_csv(run_dir / "test_predictions.csv", index=False)
                write_json(target, metrics)
                rows.append({"experiment": spec["name"], "seed": seed, **metrics})
            finally:
                del network, bank, loader, banks
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    result = pd.DataFrame(rows)
    result.to_csv(OUT_PUT_DIR / "ablation_test_results.csv", index=False)
    return result


def summarize_ablation(split="val"):
    rows, paired = [], []
    for spec in SELECTED_ABLATIONS:
        for seed in TRAIN_SEEDS:
            run_dir = RUNS_DIR / spec["name"] / f"seed_{seed}"
            path = run_dir / f"{split}_metrics.json"
            if not path.is_file():
                continue
            rows.append({"experiment": spec["name"], "seed": seed, **read_json(path)})
            reference = RUNS_DIR / "visual_only" / f"seed_{seed}" / f"{split}_predictions.csv"
            alternative = run_dir / f"{split}_predictions.csv"
            if reference.is_file() and alternative.is_file():
                paired.append({"experiment": spec["name"], "seed": seed,
                    **paired_f1_interval(pd.read_csv(reference), pd.read_csv(alternative), BOOTSTRAP_REPEATS, seed)})
    results = pd.DataFrame(rows)
    results.to_csv(OUT_PUT_DIR / f"ablation_{split}_results.csv", index=False)
    if not results.empty:
        aggregate = results.groupby("experiment").agg(
            runs=("seed", "count"), accuracy_mean=("accuracy", "mean"),
            macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"))
        aggregate.to_csv(OUT_PUT_DIR / f"ablation_{split}_mean_std.csv")
    pd.DataFrame(paired).to_csv(OUT_PUT_DIR / f"ablation_{split}_paired_vs_visual.csv", index=False)
    return results

In [ ]:
if EPOCHS < 1 or PATIENCE < 1 or BOOTSTRAP_REPEATS < 0:
    raise ValueError("Invalid epochs, patience, or bootstrap count")
if CONTEXT_TOP_K != 1:
    raise ValueError("This controlled ablation uses exactly one selected context per query")
ensure_dinov3_assets()
weights_path = resolve_dinov3_weights()
protocol = {
    "schema": "dtd_text_ablation_v1", "source_notebook_sha256": SOURCE_NOTEBOOK_SHA256,
    "helper_source_sha256": HELPER_SOURCE_SHA256, "dtd_split": DTD_SPLIT, "pool_seed": SEED,
    "split_image_ids": {split: frame.image_id.astype(str).tolist() for split, frame in SPLIT_FRAMES.items()},
    "candidate_ids": CANDIDATE_IDS, "class_names": class_names, "caption_cache": TEXT_CACHE["fingerprint"],
    "caption_csv_sha256": hashlib.sha256(CAPTION_CSV.read_bytes()).hexdigest(),
    "weights_path": str(weights_path), "weights_bytes": Path(weights_path).stat().st_size,
    "weights_mtime_ns": Path(weights_path).stat().st_mtime_ns,
    "preprocessing": {"train": repr(train_transform), "eval": repr(eval_transform)},
    "training": {name: globals()[name] for name in (
        "DINOV3_MODEL_NAME", "FEATURE_DIM", "IMAGE_SIZE", "CONTEXT_TOP_K", "CANDIDATE_POOL_FRACTION",
        "EPOCHS", "BATCH_SIZE", "NUM_WORKERS", "BACKBONE_LR", "HEAD_LR", "TACS_SELECTOR_LR", "BACKBONE_WEIGHT_DECAY",
        "HEAD_WEIGHT_DECAY", "TACS_SELECTOR_WEIGHT_DECAY", "LABEL_SMOOTHING", "TOKEN_HEAD_DROPOUT",
        "FREEZE_BACKBONE_EPOCHS", "WARMUP_EPOCHS", "PATIENCE", "GRAD_CLIP_NORM", "SUPCON_WEIGHT", "SUPCON_TEMPERATURE",
        "TACS_POLICY_WEIGHT", "TACS_ENTROPY_WEIGHT", "TACS_SELECTOR_TEMPERATURE",
        "QUERY_CONTEXT_GATE_DROPOUT", "QUERY_CONTEXT_GATE_BIAS", "CAPTION_SEMANTIC_THRESHOLD")},
    "selector_sampling": "categorical_policy_on_visual_z_plus_optional_semantic_z",
    "checkpoint_selection": "minimum_validation_cross_entropy", "test_hyperparameter_tuning": False,
}
PROTOCOL_FINGERPRINT = stable_hash(protocol)
protocol_path = OUT_PUT_DIR / "protocol.json"
if protocol_path.is_file() and read_json(protocol_path) != protocol:
    raise ValueError("Existing output uses another protocol; choose a new DTD_ABLATION_OUTPUT directory")
write_json(protocol_path, protocol)
write_json(OUT_PUT_DIR / "environment.json", {"torch": torch.__version__, "numpy": np.__version__,
           "pandas": pd.__version__, "device": str(DEVICE), "transformers": _transformers_version,
           "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None})
plan = {"protocol": PROTOCOL_FINGERPRINT, "experiments": SELECTED_ABLATIONS, "seeds": TRAIN_SEEDS}
plan_path = OUT_PUT_DIR / "experiment_plan.json"
if (OUT_PUT_DIR / "locked_test_plan.json").is_file() and plan_path.is_file() and read_json(plan_path) != plan:
    raise ValueError("Cannot extend or modify the plan after test was opened; start another experiment directory")
write_json(plan_path, plan)

AblationModel = make_ablation_model_class(MTACSModel, DinoV3TACSClassifier)
candidate_image_bank, _candidate_labels_unused, candidate_fnames = build_candidate_image_bank(
    candidate_df, eval_transform, device=torch.device(CANDIDATE_IMAGE_BANK_DEVICE), dtype=CANDIDATE_IMAGE_BANK_DTYPE)
del _candidate_labels_unused  # Candidate labels do not enter any model branch or selection function.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("Run plan:", len(SELECTED_ABLATIONS), "configurations x", len(TRAIN_SEEDS), "seeds;")
print("Maximum training epochs:", EPOCHS * len(SELECTED_ABLATIONS) * len(TRAIN_SEEDS), "(early stopping enabled)")
print("One best ViT-L checkpoint is retained per run; active last.pt also contains optimizer state.")
print("Output:", OUT_PUT_DIR, "| Candidate bank:", tuple(candidate_image_bank.shape))
display(pd.DataFrame(SELECTED_ABLATIONS))

## Probes chỉ dùng text và dấu hiệu caption bị lọc

Logistic regression fit trên train labels, báo validation: screened, unscreened, redacted, từ chỉ lớp được tìm trong caption,
và chỉ các availability mask. Các probe này không nhận ảnh; tên file/nhãn thật không được chuyển thành text.
Probe tốt cho biết text hoặc mask có tín hiệu phân loại; cần đọc cùng audit để phân biệt mô tả hợp lệ với shortcut.

In [ ]:
label_word_diagnostics = caption_label_diagnostics("val")
display(pd.DataFrame([label_word_diagnostics]))
probe_results = caption_probes()
display(probe_results)

## Train độc lập từng cấu hình; chỉ validation chọn checkpoint

In [ ]:
for spec in SELECTED_ABLATIONS:
    for training_seed in TRAIN_SEEDS:
        train_one_configuration(spec, training_seed)
display(summarize_ablation('val'))

## Intervention trên validation: cùng checkpoint, cùng ảnh context

Tắt/đảo query text, candidate text, cả hai; giữ nguyên context để kiểm tra fusion. Các dòng `reselect` cho phép đổi context
để đo cả ảnh hưởng retrieval. Gate trung bình và tỷ lệ đổi nhãn được lưu cùng delta macro-F1/CI 95%.
Redaction giữ nguyên mask caption của checkpoint gốc ở cả hai nhánh, nên chỉ thay nội dung text; nếu không còn từ,
vector text được đặt zero nhưng mask gốc vẫn được giữ cho phép can thiệp này. Đây là sensitivity test tại inference;
kết luận còn cần so với các cấu hình được train độc lập.
Mọi intervention ở cell này chỉ dùng validation, không dùng test để lựa chọn cấu hình.

In [ ]:
intervention_tables = []
if RUN_VALIDATION_INTERVENTIONS:
    for spec in SELECTED_ABLATIONS:
        if spec['name'] in {'full_both', 'full_unscreened'}:
            for training_seed in TRAIN_SEEDS:
                intervention_tables.append(validation_interventions(spec, training_seed))
if intervention_tables:
    interventions = pd.concat(intervention_tables, ignore_index=True)
    interventions.to_csv(OUT_PUT_DIR / 'validation_interventions.csv', index=False)
    display(interventions)

## Đánh giá test cuối: khóa plan trước khi tính metrics, không chọn cấu hình bằng test

In [ ]:
if RUN_FINAL_TEST:
    display(evaluate_locked_test())
else:
    print("Validation-only development: final test has not been opened.")

## Tổng hợp metrics thật, paired comparisons và báo cáo

In [ ]:
validation_summary = summarize_ablation("val")
display(validation_summary)
if RUN_FINAL_TEST:
    display(summarize_ablation("test"))

def markdown_table(frame):
    if frame.empty:
        return "No completed metrics yet."
    frame = frame.copy().round(5).fillna("")
    columns = list(frame.columns)
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    lines.extend("| " + " | ".join(str(value).replace("|", "/") for value in row) + " |"
                 for row in frame.itertuples(index=False, name=None))
    return "\n".join(lines)

report = [f"# DTD split {DTD_SPLIT}: text ablation", "", f"Protocol: `{PROTOCOL_FINGERPRINT}`", "",
          "Official train/val/test: 1880/1880/1880; candidate pool: 20% train; K=1.", "",
          "## Validation", "", markdown_table(validation_summary), "",
          "## Caption-only / availability-only validation probes", "", markdown_table(probe_results), "",
          "## Interpretation", "",
          "- Compare query_only, candidate_only and both_fusion with visual_only to separate downstream text roles.",
          "- Compare full_both with both_fusion for the semantic selector contribution.",
          "- Compare full_both with full_shuffle_both; masks are preserved, so availability is not the shuffled variable.",
          "- Compare full_unscreened with full_redacted: their eligibility masks are identical.",
          "- Inspect val_interventions.csv: fixed-context perturbations isolate fusion; reselect also changes retrieval.",
          "- A strong term-only probe indicates a class-word shortcut; it does not establish access to test annotations.",
          "- DTD category names are valid texture attributes; losing accuracy after redaction is not alone proof of leakage.",
          "- Paired confidence intervals resample images within classes. They do not estimate training-seed variability.",
          "- A single training seed/split is exploratory; use multiple seeds and official splits for robust benchmark claims."]
report += ["", "## Caption audit / label-word diagnostic", "", markdown_table(audit_summary.reset_index()), "",
           markdown_table(pd.DataFrame([label_word_diagnostics]))]
for name, title in (("ablation_val_paired_vs_visual.csv", "Paired validation vs visual baseline"),
                    ("validation_interventions.csv", "Validation interventions")):
    path = OUT_PUT_DIR / name
    if path.is_file() and path.stat().st_size > 2:
        report += ["", "## " + title, "", markdown_table(pd.read_csv(path))]
if RUN_FINAL_TEST:
    report += ["", "## Locked final test", "", markdown_table(pd.read_csv(OUT_PUT_DIR / "ablation_test_results.csv"))]
(OUT_PUT_DIR / "ablation_report.md").write_text("\n".join(report) + "\n", encoding="utf-8")

fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
for axis, metric in zip(axes, ("accuracy", "macro_f1")):
    aggregate = validation_summary.groupby("experiment")[metric].agg(["mean", "std"]).reindex(ABLATION_NAMES)
    axis.bar(aggregate.index, aggregate["mean"], color="#217c75", yerr=aggregate["std"].fillna(0), capsize=3)
    axis.set_ylabel("Validation " + metric)
    axis.set_ylim(0, 1)
    axis.tick_params(axis="x", labelrotation=70)
fig.savefig(OUT_PUT_DIR / "ablation_validation.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved report:", OUT_PUT_DIR / "ablation_report.md")